<a href="https://colab.research.google.com/github/AnirudhSekar/RKSC/blob/main/rksc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reasoning-KV Shared Cache (RKSC)


In [ ]:
# Colab setup — recommended runtime: A100 (40 GB) or L4 (22 GB).
# Uncomment on a fresh environment:
# !pip install -q "transformers>=4.40" datasets tqdm numpy scipy matplotlib seaborn pandas

import os
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')


In [ ]:
import sys, os, gc, json, time, logging, warnings
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from scipy import stats
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

logging.basicConfig(level=logging.WARNING)

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.set_float32_matmul_precision("high")
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    _p = torch.cuda.get_device_properties(0)
    print(f'GPU : {_p.name}  ({_p.total_memory/1e9:.0f} GB)')

import transformers
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')

_RAM_CACHE: dict = {}

def ckpt_save(name: str, obj): _RAM_CACHE[name] = obj
def ckpt_load(name: str, default=None): return _RAM_CACHE.get(name, default)
def ckpt_exists(name: str) -> bool: return name in _RAM_CACHE

def cpu_ram_gb() -> float:
    # Local import: psutil is optional; cpu_ram_gb is only called
    # opportunistically and we don't want to require psutil at module import.
    import psutil
    return psutil.Process().memory_info().rss / 1e9

print('Imports ready.')


# ─────────────────────────────────────────────────────────────────────────
# Shared helpers — factored out of duplicated definitions across §6–§11.
# ─────────────────────────────────────────────────────────────────────────
_PAD_FILLER = (
    'Shared context: state all assumptions explicitly and '
    'preserve dimensional units throughout each step. '
)
_PAD_TEMPLATE = 'Solve step by step:\n{q}\nLet us reason step by step.'


def pad_to_target(problem: str, tokenizer, target: int,
                   max_seq_len: int = 2048,
                   template: str = _PAD_TEMPLATE,
                   filler: str = _PAD_FILLER) -> str:
    """Pad problem text with neutral filler until the templated prompt
    reaches the target token count.  One bulk prepend, then a small
    fine-tune loop — no per-token loops."""
    def _n_tok(text: str) -> int:
        return int(tokenizer(text, return_tensors='pt',
                             truncation=True, max_length=max_seq_len
                             )['input_ids'].shape[1])
    hard_cap = max_seq_len - 256
    target_c = min(target, hard_cap)
    aug      = problem.strip()
    current  = _n_tok(template.format(q=aug))
    if current >= target_c:
        return aug
    filler_toks = max(_n_tok(filler), 1)
    n_reps      = max(1, int(np.ceil((target_c - current) / filler_toks)))
    aug         = filler * n_reps + aug
    for _ in range(20):
        if _n_tok(template.format(q=aug)) >= target_c:
            break
        aug = filler + aug
    return aug


def bootstrap_ci_ratio(lat_base, lat_sys, n_boot: int = 2000,
                        seed: int = 42, alpha: float = 0.05):
    """95% bootstrap CI on ratio-of-means speedup = mean(base)/mean(sys).

    Each array is resampled with its own length so mismatched list sizes
    (e.g. when one condition produces fewer timing entries) do not crash.
    """
    rng  = np.random.default_rng(seed)
    base = np.asarray(lat_base, dtype=float)
    sys_ = np.asarray(lat_sys,  dtype=float)
    nb_  = len(base)
    ns_  = len(sys_)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = (base[rng.integers(0, nb_, nb_)].mean() /
                    sys_[rng.integers(0, ns_, ns_)].mean())
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi


def cv_percent(xs) -> float:
    """Coefficient of variation in percent — measurement reliability proxy."""
    a = np.asarray(xs, dtype=float)
    return float(a.std() / a.mean() * 100) if a.mean() != 0 else 0.0


def paired_ttest(a, b):
    """Paired t-test on two latency lists; aligns lengths by truncating the
    longer one so mismatched lists do not crash scipy.stats.ttest_rel."""
    n = min(len(a), len(b))
    if n < 2:
        return float('nan'), float('nan')
    return stats.ttest_rel(a[:n], b[:n])


## §1  Configuration

In [ ]:
CALIBRATED = {
    'Qwen/Qwen2.5-7B-Instruct':           {'tau': 0.82, 'theta': 8.0},
    'mistralai/Mistral-7B-Instruct-v0.3': {'tau': 0.80, 'theta': 0.50},
    'tiiuae/Falcon3-7B-Instruct':         {'tau': 0.80, 'theta': 8.0},
    'tiiuae/Falcon3-10B-Instruct':        {'tau': 0.80, 'theta': 8.0},
    'meta-llama/Meta-Llama-3-8B-Instruct':     {'tau': 0.82, 'theta': 8.0},
}

MODEL_VRAM_GB = {
    'Qwen/Qwen2.5-7B-Instruct':           15.0,
    'mistralai/Mistral-7B-Instruct-v0.3': 15.0,
    'tiiuae/Falcon3-7B-Instruct':         15.0,
    'tiiuae/Falcon3-10B-Instruct':        20.0,
    'meta-llama/Meta-Llama-3-8B-Instruct':     15.0,
}

MODEL_ID         = 'Qwen/Qwen2.5-7B-Instruct'
N_PROBLEMS       = 30
N_RUNS           = 3
N_WARMUP         = 2      # warmup solves per condition → reduces RKSC CV
B_FACTOR         = 8
MAX_DEPTH        = 3
MAX_NEW_TOK      = 32
MAX_NEW_TOK_EXT  = 8
ATTN_IMPL        = 'sdpa'
MAX_SEQ_LEN      = 2048

CKPT_VERSION = (
    f'{MODEL_ID}|tok={MAX_NEW_TOK}|B={B_FACTOR}'
    f'|N={N_PROBLEMS}|pfx1024|runs={N_RUNS}|warm={N_WARMUP}|v5'
)

print(f'Model       : {MODEL_ID}')
print(f'N / B / tok : {N_PROBLEMS} / {B_FACTOR} / {MAX_NEW_TOK}')
print(f'Warmup      : {N_WARMUP} solves per condition')


In [ ]:
_BENCH_KEYS = ['lat_batch_nokv', 'lat_kv', 'lat_cgee', 'lat_rksc', 'extended_results']

_stored_sig = _RAM_CACHE.get('_ckpt_version', '')
if _stored_sig != CKPT_VERSION:
    print(f'Config changed — clearing cached results.')
    for _k in _BENCH_KEYS:
        _RAM_CACHE.pop(_k, None)
        if _k in globals(): del globals()[_k]
    _RAM_CACHE['_ckpt_version'] = CKPT_VERSION
else:
    print(f'Config unchanged. Cached keys: {[k for k in _BENCH_KEYS if k in _RAM_CACHE]}')


## §2  Dataset & Model Loading

In [ ]:
from huggingface_hub import login, whoami

_hf_token = None

try:
    from google.colab import userdata as _ud
    _tok = _ud.get('HF_TOKEN')
    if _tok: _hf_token = _tok.strip()
except (ImportError, ModuleNotFoundError):
    # Not running on Colab — fall through to env var.
    pass
except Exception as _e:
    print(f'Colab userdata lookup failed: {_e}')

if not _hf_token:
    _hf_token = os.environ.get('HF_TOKEN', '').strip() or None

if _hf_token:
    try:
        login(_hf_token, add_to_git_credential=False)
        print(f'HF login OK — {whoami()["name"]}')
    except Exception as e:
        print(f'HF login failed: {e}')
        _hf_token = None
else:
    print('No HF_TOKEN — GPQA requires a token (https://huggingface.co/settings/tokens)')


In [ ]:
_GPQA_PREAMBLE = (
    'You are solving a graduate-level science problem. '
    'Before answering: (1) identify relevant physical laws; '
    '(2) define all symbols and state assumptions; '
    '(3) set up the mathematical framework and work through each step; '
    '(4) check dimensional consistency; (5) state the final answer with units.\n\n'
)

if not _hf_token:
    raise RuntimeError(
        'HF_TOKEN required. Set it as a Colab secret or environment variable, '
        'then re-run the login cell.'
    )

print('Loading GPQA Diamond ...')
_ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train', token=_hf_token)
_ds = _ds.select(range(min(N_PROBLEMS, len(_ds))))
gpqa_probs = [_GPQA_PREAMBLE + ex['Question'] for ex in _ds]
gpqa_ans   = [ex['Correct Answer'] for ex in _ds]
print(f'  Loaded {len(gpqa_probs)} problems')
_problems = gpqa_probs  # alias used by §8–§12 experiment cells


## §3  Core Dataclasses

In [ ]:
@dataclass
class RKSCConfig:
    tau_similarity_threshold: float = 0.75
    theta_entropy_threshold:  float = 12.0
    cgee_min_exit_layer:      int   = 2
    cgee_stability_eps:       float = 3.0
    cgee_gen_conf_threshold:  float = 0.70
    cgee_gen_conf_gap:        float = 0.06
    cgee_use_relative_gap:    bool  = True
    cgee_relative_gap_thresh: float = 0.08
    cgee_early_exit_decode:   bool  = True
    cgee_min_decode_steps:    int   = 3
    cgee_decode_conf_thresh:  float = 0.72
    cgee_decode_rel_gap:      float = 0.10
    # PRISM fallback disabled — always use clean ASKS path.
    # PRISM runs an extra full forward on SDPA and is never faster.
    prism_enabled:            bool  = False


@dataclass
class RSBCMConfig:
    max_blocks:              int   = 2000
    cache_capacity_fraction: float = 0.80


@dataclass
class RKSCSolverConfig:
    rksc:  RKSCConfig  = field(default_factory=RKSCConfig)
    rsbcm: RSBCMConfig = field(default_factory=RSBCMConfig)
    branching_factor:        int  = B_FACTOR
    max_depth:               int  = MAX_DEPTH
    max_new_tokens_per_step: int  = MAX_NEW_TOK
    verify_with_cgee:        bool = True
    verify_full_depth:       bool = False
    max_seq_len:             int  = MAX_SEQ_LEN
    batch_without_kv:        bool = False


print('Config dataclasses loaded.')


## §4  Model Loading

In [ ]:
def load_model_and_tokenizer(model_id: str, attn_impl: str = 'sdpa'):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    dtype = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32

    if attn_impl == 'flash_attention_2':
        import importlib
        if importlib.util.find_spec('flash_attn') is None:
            print('FlashAttention2 not found -> falling back to SDPA')
            attn_impl = 'sdpa'

    t0 = time.perf_counter()
    if DEVICE.type == 'cuda':
        m = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype         = dtype,
            device_map          = 'cuda:0',
            attn_implementation = attn_impl,
        )
    else:
        m = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=dtype,
            low_cpu_mem_usage=True,
            attn_implementation=attn_impl)
        m = m.to(DEVICE)
    m.eval()

    if hasattr(m, 'generation_config'):
        m.generation_config.temperature = None
        m.generation_config.top_p       = None
        m.generation_config.top_k       = None
        m.generation_config.do_sample   = False

    elapsed = time.perf_counter() - t0
    if DEVICE.type == 'cuda':
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        used_gb = m.get_memory_footprint() / 1e9
        print(f'  Weights : {used_gb:.1f} GB  GPU free after load: {free_gb:.1f} GB  ({elapsed:.0f}s)')
    print(f'  Loaded  : {model_id} ({dtype}  attn={attn_impl})')
    return m, tok


def diagnose_architecture(model, tokenizer) -> Dict:
    dev = model.get_input_embeddings().weight.device
    inp = tokenizer('hello world', return_tensors='pt').to(dev)
    with torch.no_grad():
        out = model(**inp, output_hidden_states=True)
    hs_ok = (hasattr(out, 'hidden_states') and out.hidden_states is not None
             and len(out.hidden_states) > 1)
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layer_path, n_layers = 'model.layers', len(model.model.layers)
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        layer_path, n_layers = 'transformer.h', len(model.transformer.h)
    else:
        layer_path = 'unknown'
        n_layers   = getattr(model.config, 'num_hidden_layers',
                             getattr(model.config, 'n_layer', 12))
    hidden_size = getattr(model.config, 'hidden_size',
                          getattr(model.config, 'n_embd', 768))
    n_heads = getattr(model.config, 'num_attention_heads',
                      getattr(model.config, 'n_head', None))
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    head_dim = getattr(model.config, 'head_dim', None)
    if head_dim is None and hidden_size and n_heads:
        try:
            head_dim = hidden_size // int(n_heads)
        except Exception:
            head_dim = None
    max_pos = getattr(model.config, 'max_position_embeddings',
                      getattr(model.config, 'seq_length', None))
    return dict(hs_in_model_out=hs_ok, n_layers=n_layers,
                hidden_size=hidden_size, vocab_size=model.config.vocab_size,
                layer_path=layer_path,
                num_attention_heads=n_heads,
                num_key_value_heads=n_kv_heads,
                head_dim=head_dim,
                max_position_embeddings=max_pos)


def infer_eval_seq_len(model, tokenizer, arch: Dict, requested: Optional[int] = None) -> int:
    candidates: List[int] = []
    for cand in [
        requested,
        arch.get('max_position_embeddings'),
        getattr(model.config, 'max_position_embeddings', None),
        getattr(model.config, 'seq_length', None),
        getattr(tokenizer, 'model_max_length', None),
    ]:
        if isinstance(cand, (int, np.integer)) and 0 < int(cand) < 100000:
            candidates.append(int(cand))
    if candidates:
        return min(candidates)
    return 2048

print('Model utilities loaded.')


def unload_model(model=None, tokenizer=None, arch=None):
    if arch is not None:
        arch.pop('_unembed_weight', None)
    for obj in [model, tokenizer]:
        if obj is not None:
            del obj
    global _CGEE_EXIT_HISTORY
    _CGEE_EXIT_HISTORY = []
    for _ in range(5):
        gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
        print(f'  VRAM free after unload: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')


In [ ]:
for _v in ['model', 'tokenizer', 'ARCH']:
    if _v in globals():
        del globals()[_v]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'VRAM before load: {(_total-_free)/1e9:.1f} / {_total/1e9:.0f} GB used')
    _needed = MODEL_VRAM_GB.get(MODEL_ID, 16.0)
    if _free/1e9 < _needed:
        raise RuntimeError(f'Need ~{_needed:.0f} GB free, have {_free/1e9:.1f} GB')

print(f'Loading {MODEL_ID} ...')
model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
ARCH = diagnose_architecture(model, tokenizer)
ARCH['_unembed_weight'] = model.get_output_embeddings().weight.data
MAX_SEQ_LEN = infer_eval_seq_len(model, tokenizer, ARCH)

print(f'  Layers={ARCH["n_layers"]}  hidden={ARCH["hidden_size"]}  vocab={ARCH["vocab_size"]}')
if torch.cuda.is_available():
    print(f'  VRAM free after load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')


## §5  RKSC Solver

```
solve(problem)
├── _root_pass()        → prefix KV cache (1 forward, batch=1)
├── _branch_passes()    → B branches decoded together (batched)
│   ├── _expand_kv(B)   → repeat_interleave KV to batch B
│   └── _stepwise_decode() → token-by-token decode loop
└── _verify()           → batched YES/NO scoring forward pass
    └── _verify_smart() → CGEE gate: skip verify when confident
```


In [ ]:
@dataclass
class BranchSimilarityRecord:
    branch_id:  int
    similarity: float
    kv_reused:  bool


@dataclass
class EarlyExitResult:
    exit_layer:    Optional[int]
    entropy_curve: List[float]
    exited_early:  bool


@dataclass
class BranchResult:
    branch_id:             int
    depth:                 int
    text:                  str
    score:                 Optional[float]
    kv_reused:             bool
    similarity:            float
    cgee_exit_layer:       Optional[int]
    latency_ms:            float
    generation_confidence: float = 0.5
    decode_exit_step:      int   = -1
    problem:               str   = ''


@dataclass
class SolveResult:
    problem:             str
    answer:              str
    branches:            List[BranchResult]
    root_latency_ms:     float
    total_latency_ms:    float
    nowaste_latency_ms:  float

    @property
    def speedup(self) -> float:
        if self.nowaste_latency_ms <= 0:
            return 1.0
        return self.nowaste_latency_ms / self.total_latency_ms

    @property
    def kv_reuse_rate(self) -> float:
        if not self.branches:
            return 0.0
        return sum(b.kv_reused for b in self.branches) / len(self.branches)

    @property
    def mean_cgee_exit_layer(self) -> Optional[float]:
        ls = [b.cgee_exit_layer for b in self.branches if b.cgee_exit_layer is not None]
        return float(np.mean(ls)) if ls else None


print('Data structures loaded.')


In [ ]:
def _squeeze_hs(h):
    h = h.float().nan_to_num(0.0)
    if h.dim() == 3:
        h = h[:, -1, :]
    if h.dim() == 2:
        h = h[0]
    return h


class ASKSManager:
    def __init__(self, cfg: RKSCConfig, arch: Dict):
        self.cfg  = cfg
        self.arch = arch
        self._root_hs_stacked: Optional[torch.Tensor] = None
        self._root_kv = None
        self.records: Dict[int, BranchSimilarityRecord] = {}

    def capture_root(self, kv_cache=None, hidden_states=None):
        if hidden_states:
            vecs = [_squeeze_hs(h).detach() for h in hidden_states]
            stacked = torch.stack(vecs)
            # Pre-normalise root hidden states once so compute_similarity
            # avoids redundant norm ops on each of the B branch comparisons.
            norms = stacked.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            self._root_hs_stacked     = stacked
            self._root_hs_normalised  = stacked / norms   # cached unit vectors
        else:
            self._root_hs_stacked    = None
            self._root_hs_normalised = None
        self._root_kv = kv_cache

    def compute_similarity(self, branch_hs=None):
        if self._root_hs_stacked is None or not branch_hs:
            return 0.0
        dev = self._root_hs_stacked.device
        branch_stacked = torch.stack([_squeeze_hs(h).to(dev) for h in branch_hs])
        # Use pre-normalised root (computed once in capture_root) to avoid
        # redundant norm ops across B branch comparisons.
        if hasattr(self, '_root_hs_normalised') and self._root_hs_normalised is not None:
            root_unit = self._root_hs_normalised.to(dev)
        else:
            norms = self._root_hs_stacked.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            root_unit = self._root_hs_stacked / norms
        b_norms = branch_stacked.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        branch_unit = branch_stacked / b_norms
        cos = (root_unit * branch_unit).sum(-1)  # [n_layers]
        # Weight later layers more heavily (exponential schedule)
        n = cos.shape[0]
        weights = torch.exp(torch.linspace(0.0, 1.5, n, device=dev))
        weights = weights / weights.sum()
        return float((cos * weights).sum().item())

    def score_branch(self, branch_id: int,
                     branch_attn: Optional[List] = None,
                     branch_hs:   Optional[List[torch.Tensor]] = None) -> BranchSimilarityRecord:
        sim   = self.compute_similarity(branch_hs)
        reuse = sim >= self.cfg.tau_similarity_threshold
        rec   = BranchSimilarityRecord(branch_id, sim, reuse)
        self.records[branch_id] = rec
        return rec

    def score_branches_batched(self, branch_attns, branch_hss, branch_ids):
        return [self.score_branch(bid, ba, bh)
                for bid, ba, bh in zip(branch_ids, branch_attns, branch_hss)]

    def get_root_kv(self, branch_id: int):
        rec = self.records.get(branch_id)
        if rec is not None and rec.kv_reused and self._root_kv is not None:
            return self._root_kv
        return None

    def compute_per_layer_similarity(self, branch_hs) -> torch.Tensor:
        """Per-layer cosine similarity: root vs branch. Returns [n_layers] on CPU."""
        if self._root_hs_stacked is None or not branch_hs:
            return torch.zeros(1)
        dev = self._root_hs_stacked.device
        b_stacked = torch.stack([_squeeze_hs(h).to(dev) for h in branch_hs])
        n_r = self._root_hs_stacked.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        n_b = b_stacked.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        return ((self._root_hs_stacked / n_r) * (b_stacked / n_b)).sum(-1).cpu()

    def find_divergence_depth(self, per_layer_sims: torch.Tensor, tau: float) -> int:
        """Deepest layer index where cosine similarity >= tau. Returns 0 if none pass."""
        passing = (per_layer_sims >= tau).nonzero(as_tuple=False)
        return int(passing[-1].item()) if len(passing) > 0 else 0

    def reset(self):
        self._root_hs_stacked = None
        self._root_kv         = None
        self.records          = {}


print('ASKS loaded.')


In [ ]:
class RSBCMManager:
    @dataclass
    class Block:
        block_id:   int
        tree_depth: int
        branch_id:  int
        importance: float

        @property
        def eviction_priority(self) -> float:
            return self.importance / (self.tree_depth + 1)

    def __init__(self, cfg: RSBCMConfig):
        self.cfg   = cfg
        self._pool: Dict[int, RSBCMManager.Block] = {}
        self._next_id        = 0
        self.eviction_events = 0

    def allocate(self, tree_depth: int, branch_id: int,
                 importance: float = 1.0) -> int:
        bid = self._next_id
        self._next_id += 1
        self._pool[bid] = RSBCMManager.Block(bid, tree_depth, branch_id, importance)
        self._maybe_evict()
        return bid

    def _maybe_evict(self):
        if len(self._pool) <= self.cfg.max_blocks:
            return
        n_evict = len(self._pool) - self.cfg.max_blocks
        ordered = sorted(self._pool.values(), key=lambda b: b.eviction_priority)
        for blk in ordered[:n_evict]:
            del self._pool[blk.block_id]
            self.eviction_events += 1

    def evict_branch(self, branch_id: int) -> int:
        victims = [b for b in self._pool.values() if b.branch_id == branch_id]
        for v in victims:
            del self._pool[v.block_id]
        return len(victims)

    @property
    def n_blocks(self) -> int:
        return len(self._pool)

    def reset(self):
        self._pool        = {}
        self._next_id     = 0
        self.eviction_events = 0


print('RSBCM loaded.')


In [ ]:
class CGEEAnalyzer:
    def __init__(self, cfg: RKSCConfig, arch: Dict):
        self.cfg  = cfg
        self.arch = arch

    def analyze(self, model_output) -> EarlyExitResult:
        if not (hasattr(model_output, 'hidden_states')
                and model_output.hidden_states is not None):
            return EarlyExitResult(None, [], False)

        unembed = self.arch.get('_unembed_weight')
        if unembed is None:
            return EarlyExitResult(None, [], False)

        all_hs   = [h for h in model_output.hidden_states if h is not None]
        layer_hs = all_hs[1:] if len(all_hs) > 1 else all_hs

        raw:       List[torch.Tensor] = []
        valid_idx: List[int]          = []
        for i, hs in enumerate(layer_hs):
            h = hs[:, -1, :].float().nan_to_num(0.0)
            if h.norm() > 0:
                raw.append(h.squeeze(0))
                valid_idx.append(i)

        if not raw:
            return EarlyExitResult(None, [], False)

        dev       = unembed.device
        all_h     = torch.stack(raw).to(dev)
        unembed_f = unembed.float()
        logits    = F.linear(all_h, unembed_f)
        logits    = logits.clamp(-1000.0, 1000.0)
        probs     = logits.softmax(dim=-1)
        entropies = -(probs * (probs + 1e-10).log()).sum(dim=-1)
        ent_list  = entropies.tolist()

        entropy_curve: List[float] = []
        exit_layer:    Optional[int] = None

        for layer_idx, ent in zip(valid_idx, ent_list):
            if not np.isfinite(ent):
                continue
            entropy_curve.append(ent)
            n = len(entropy_curve)
            if (exit_layer is None
                    and layer_idx >= self.cfg.cgee_min_exit_layer
                    and ent < self.cfg.theta_entropy_threshold
                    and n >= 2
                    and abs(ent - entropy_curve[-2]) < self.cfg.cgee_stability_eps):
                exit_layer = layer_idx

        return EarlyExitResult(
            exit_layer    = exit_layer,
            entropy_curve = entropy_curve,
            exited_early  = exit_layer is not None,
        )


print('CGEE loaded.')


In [ ]:
_BRANCH_HINTS = [
    # Richer, step-directive hints that give the model concrete structure
    "Use first principles. Identify the key physical law, define all variables "
    "with units, then derive step by step.",
    "Apply conservation laws and symmetry. State which quantity is conserved, "
    "write the conservation equation, and solve.",
    "Work backwards from the answer choices. Substitute each option and check "
    "which satisfies the governing equation.",
    "Set up a full mathematical model: list knowns, unknowns, and equations, "
    "then solve the system algebraically.",
    "Use a limiting case (e.g. take a parameter → 0 or → ∞) to simplify, "
    "solve the limit, then generalise.",
    "Draw a free-body or circuit diagram mentally. Identify all forces/flows, "
    "write equilibrium or Kirchhoff equations, solve.",
    "Use dimensional analysis and order-of-magnitude estimation to constrain "
    "the answer before computing exactly.",
    "Apply the most directly relevant formula. State the formula, identify "
    "every term, substitute values, and simplify.",
]

_CGEE_EXIT_HISTORY: list = []
_KV_PROBE_VERBOSE = True   # set False in extended eval

def _ms(t0: float) -> float:
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000


class _EarlyExitSignal(Exception):
    def __init__(self, layer_idx: int):
        self.layer_idx = layer_idx


class _HSWrapper:
    """Minimal stand-in for a model output with only .hidden_states set."""
    __slots__ = ('hidden_states',)
    def __init__(self, hs): self.hidden_states = hs


class RKSCSolverV8:
    """
    RKSC solver: shared prefix KV cache + step-wise batched decoding.

    Core design:
    - Prefix KV computed once; expanded to B branches via repeat_interleave.
    - Branches process only suffix tokens, sharing all GPU decode kernels.
    - CGEE gates the verify pass on generation confidence.
    """

    def __init__(self, model, tokenizer, cfg: RKSCSolverConfig,
                 arch: Dict, disable_reuse: bool = False):
        self.model         = model
        self.tok           = tokenizer
        self.cfg           = cfg
        self.arch          = arch
        self.disable_reuse = disable_reuse
        self._dev          = model.get_input_embeddings().weight.device
        self._max_seq_len  = infer_eval_seq_len(
            model, tokenizer, arch, getattr(cfg, 'max_seq_len', None)
        )

        if '_unembed_weight' not in arch:
            arch['_unembed_weight'] = model.get_output_embeddings().weight.data

        self.asks  = ASKSManager(cfg.rksc, arch)
        self.rsbcm = RSBCMManager(cfg.rsbcm)
        self.cgee  = CGEEAnalyzer(cfg.rksc, arch)
        self._root_layer_importances: List[float] = []
        self._root_pkv_obj   = None
        self._prefix_len     = 0
        self._n_solved       = 0

        self._cgee_handles: list = []

        _yn = self.tok
        self._yes_id: Optional[int] = next(
            (ids[0] for ids in [_yn.encode(' YES', add_special_tokens=False),
                                _yn.encode('YES',  add_special_tokens=False)] if ids),
            None)
        self._no_id:  Optional[int] = next(
            (ids[0] for ids in [_yn.encode(' NO',  add_special_tokens=False),
                                _yn.encode('NO',   add_special_tokens=False)] if ids),
            None)

    def _get_transformer_layers(self, layer_indices=None):
        lp = self.arch.get('layer_path', '')
        if lp == 'model.layers' and hasattr(self.model, 'model'):
            all_layers = list(self.model.model.layers)
        elif lp == 'transformer.h' and hasattr(self.model, 'transformer'):
            all_layers = list(self.model.transformer.h)
        else:
            all_layers = [m for n, m in self.model.named_modules()
                          if n.count('.') == 2 and any(k in n for k in ('layers', '.h'))]
        if layer_indices is not None:
            return [all_layers[i] for i in layer_indices if i < len(all_layers)]
        return all_layers


    def remove_hooks(self):
        """Remove all CGEE hooks. Call before discarding a solver."""
        for h in self._cgee_handles:
            h.remove()
        self._cgee_handles.clear()

    def _tok(self, text: str, **kwargs):
        return self.tok(text, return_tensors='pt',
                        truncation=True, max_length=self._max_seq_len,
                        **kwargs).to(self._dev)

    def _shared_prefix(self, problem: str) -> str:
        # Richer prefix: gives the model explicit reasoning structure
        # that all branches inherit, improving branch quality uniformly.
        return (
            'You are an expert scientist and mathematician. '
            'Reason carefully and show all working.\n\n'
            f'{problem}\n\n'
            'Think through this systematically:'
        )

    def _branch_suffix(self, branch_idx: int) -> str:
        hint = _BRANCH_HINTS[branch_idx % len(_BRANCH_HINTS)]
        return f'\nApproach #{branch_idx + 1}: {hint}\nStep 1:'

    def _expand_kv(self, pkv, B: int):
        """
        Expand a batch-1 KV cache to batch B via repeat_interleave.
        Uses .clone() on each tensor — faster than deepcopy, avoids
        expand() which can trigger contiguity issues in attention kernels.
        """
        if pkv is None:
            return None
        from transformers import DynamicCache
        try:
            new_cache = DynamicCache()
            if hasattr(pkv, 'key_cache') and len(pkv.key_cache) > 0:
                for li in range(len(pkv.key_cache)):
                    new_cache.update(
                        pkv.key_cache[li].repeat_interleave(B, dim=0),
                        pkv.value_cache[li].repeat_interleave(B, dim=0),
                        layer_idx=li)
            elif hasattr(pkv, 'layers') and len(pkv.layers) > 0:
                for li, layer in enumerate(pkv.layers):
                    new_cache.update(
                        layer.keys.repeat_interleave(B, dim=0),
                        layer.values.repeat_interleave(B, dim=0),
                        layer_idx=li)
            else:
                return deepcopy(pkv)
            return new_cache
        except Exception as _e:
            print(f'[V8] _expand_kv: {_e}', flush=True)
            return deepcopy(pkv)

    def _kv_seq_len(self, pkv) -> int:
        """Return number of tokens currently in the KV cache."""
        if pkv is None:
            return 0
        if hasattr(pkv, 'key_cache') and pkv.key_cache:
            return pkv.key_cache[0].shape[2]
        if isinstance(pkv, tuple) and pkv and pkv[0]:
            return pkv[0][0].shape[2]
        return 0

    def _stepwise_decode(self,
                         input_ids:       torch.Tensor,
                         attention_mask:  torch.Tensor,
                         past_key_values,
                         max_new_tokens:  int,
                         early_exit_cfg:  Optional[tuple] = None,
                         ) -> Tuple[torch.Tensor, float]:
        """Decode using a pre-filled KV cache via a direct model() loop."""
        B      = input_ids.shape[0]
        dev    = self._dev
        eos_id = self.tok.eos_token_id or 0

        total_ctx   = attention_mask.shape[1]
        max_buf     = total_ctx + max_new_tokens
        full_mask   = torch.ones(B, max_buf, dtype=torch.long, device=dev)
        _mask_is_dense = bool(attention_mask.all())
        if not _mask_is_dense:
            full_mask[:, :total_ctx] = attention_mask  # only copy if padding present
        generated   = torch.full((B, max_new_tokens), eos_id,
                                  dtype=torch.long, device=dev)
        finished    = torch.zeros(B, dtype=torch.bool, device=dev)

        t0         = time.perf_counter()
        pkv        = past_key_values
        curr       = total_ctx
        _conf_list: list = []  # top-1 softmax probability per step per branch

        _ee_enabled   = (early_exit_cfg is not None and B > 1)
        _ee_min_steps = early_exit_cfg[0] if _ee_enabled else max_new_tokens
        _ee_conf      = early_exit_cfg[1] if _ee_enabled else 1.1
        _ee_rel_gap   = early_exit_cfg[2] if _ee_enabled else 1.1
        _exit_step    = -1  # -1 = no early exit

        with torch.inference_mode():
            out      = self.model(
                input_ids       = input_ids,
                attention_mask  = full_mask[:, :curr],
                past_key_values = pkv,
                use_cache       = True,
            )
            next_tok = out.logits[:, -1:, :].argmax(dim=-1)   # [B, 1]
            _conf_list.append(
                out.logits[:, -1, :].float().softmax(-1).max(-1).values.cpu())
            pkv      = out.past_key_values
            generated[:, 0] = next_tok[:, 0]
            finished |= (next_tok[:, 0] == eos_id)
            curr     += 1

            for step in range(1, max_new_tokens):
                if finished.all():
                    break
                out      = self.model(
                    input_ids       = next_tok,                  # [B, 1]
                    attention_mask  = full_mask[:, :curr],       # view, no alloc
                    past_key_values = pkv,
                    use_cache       = True,
                )
                next_tok = out.logits[:, -1:, :].argmax(dim=-1)
                _conf_list.append(
                    out.logits[:, -1, :].float().softmax(-1).max(-1).values.cpu())
                pkv      = out.past_key_values
                generated[:, step] = next_tok[:, 0]
                finished |= (next_tok[:, 0] == eos_id)
                curr     += 1

                if _ee_enabled and step >= _ee_min_steps:
                    _stacked   = torch.stack(_conf_list)         # [steps, B]
                    _br_means  = _stacked.mean(0)               # [B] rolling mean
                    _sorted, _ = _br_means.sort(descending=True)
                    _top, _sec = float(_sorted[0]), float(_sorted[1])
                    _rel_lead  = (_top - _sec) / max(_top, 1e-6)
                    if _top >= _ee_conf and _rel_lead >= _ee_rel_gap:
                        finished[:] = True  # stop all branches
                        _exit_step  = step
                        break

        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        _mean_conf = (torch.stack(_conf_list).mean(0)
                      if _conf_list else torch.full((B,), 0.5))
        return generated, (time.perf_counter() - t0) * 1000, _mean_conf, _exit_step


    def _root_pass(self, problem: str):
        """Compute prefix KV cache. Probes both paths and caches the faster decision."""
        prefix_text = self._shared_prefix(problem)
        prefix_inp  = self._tok(prefix_text)
        prefix_len  = prefix_inp['input_ids'].shape[1]
        self._prefix_len = prefix_len

        if self.disable_reuse:
            self._root_pkv_obj = None
            return "", 0.0, prefix_len

        if not hasattr(self, '_kv_probe_cache'):
            self._kv_probe_cache: Dict[int, bool] = {}
        _bucket = prefix_len // 64   # re-probe if prefix changes by >64 tokens

        if _bucket not in self._kv_probe_cache:
            _sfx_text = self._branch_suffix(0)
            _sfx_inp  = self.tok(_sfx_text, return_tensors='pt',
                                  truncation=True, max_length=128).to(self._dev)
            _full_inp = self._tok(prefix_text + _sfx_text)

            _ts_full = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(**_full_inp, use_cache=True,
                               output_hidden_states=False, output_attentions=False)
                _ts_full.append(_ms(t0))
            _ms_full = min(_ts_full)

            with torch.inference_mode():
                _fwd = self.model(**prefix_inp, use_cache=True,
                                  output_hidden_states=False, output_attentions=False)
            _pkv_probe = _fwd.past_key_values; del _fwd
            _ts_pre = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    _fwd2 = self.model(**prefix_inp, use_cache=True,
                                       output_hidden_states=False, output_attentions=False)
                _ts_pre.append(_ms(t0))
                del _fwd2.past_key_values, _fwd2
            _ms_pre = min(_ts_pre)

            _attn_sfx = torch.cat([
                torch.ones(1, prefix_len, dtype=torch.long, device=self._dev),
                torch.ones(1, _sfx_inp['input_ids'].shape[1],
                           dtype=torch.long, device=self._dev),
            ], dim=1)
            _ts_sfx = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(_sfx_inp['input_ids'], attention_mask=_attn_sfx,
                               past_key_values=_pkv_probe, use_cache=True)
                _ts_sfx.append(_ms(t0))
            _ms_sfx = min(_ts_sfx)

            B = self.cfg.branching_factor


            _sfx_ids_B  = _sfx_inp['input_ids'].expand(B, -1)
            _pfx_ids_B  = prefix_inp['input_ids'].expand(B, -1)
            _full_ids_B = torch.cat([_pfx_ids_B, _sfx_ids_B], dim=1)
            _full_msk_B = torch.ones(B, _full_ids_B.shape[1],
                                     dtype=torch.long, device=self._dev)
            _ts_batch_full = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(input_ids=_full_ids_B,
                               attention_mask=_full_msk_B,
                               use_cache=False,
                               output_hidden_states=False,
                               output_attentions=False)
                _ts_batch_full.append(_ms(t0))
            _ms_batch_full = min(_ts_batch_full)

            _attn_sfx_B = torch.cat([
                torch.ones(B, prefix_len, dtype=torch.long, device=self._dev),
                torch.ones(B, _sfx_inp['input_ids'].shape[1],
                           dtype=torch.long, device=self._dev),
            ], dim=1)
            _ts_batch_sfx = []
            for _ in range(3):
                _pkv_B = self._expand_kv(_pkv_probe, B)
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(_sfx_ids_B,
                               attention_mask=_attn_sfx_B,
                               past_key_values=_pkv_B,
                               use_cache=True)
                _ts_batch_sfx.append(_ms(t0))
                del _pkv_B
            _ms_batch_sfx = min(_ts_batch_sfx)

            _kv_net     = _ms_batch_full - (_ms_pre + _ms_batch_sfx)
            _beneficial = _kv_net > 0

            _per_branch_saving = _ms_full - _ms_sfx
            if _per_branch_saving > 0:
                _breakeven_B    = _ms_pre / _per_branch_saving
                _breakeven_str  = f'breakeven_B≈{_breakeven_B:.1f}'
            else:
                _breakeven_B    = float('inf')
                _breakeven_str  = 'breakeven_B=∞ (KV load > recompute on SDPA)'

            self._kv_probe_cache[_bucket] = _beneficial
            if not hasattr(self, '_kv_probe_measurements'):
                self._kv_probe_measurements: dict = {}
            self._kv_probe_measurements[_bucket] = dict(
                prefix_len=prefix_len,
                ms_full_B1=_ms_full, ms_prefix_fwd=_ms_pre,
                ms_sfx_B1=_ms_sfx,
                ms_batch_full=_ms_batch_full, ms_batch_sfx=_ms_batch_sfx,
                per_branch_saving_ms=_per_branch_saving,
                breakeven_B=_breakeven_B, beneficial=_beneficial,
            )
            if _KV_PROBE_VERBOSE:
                print(f'  KV probe [prefix≈{prefix_len}tok B={B} bucket={_bucket}]: '
                      f'batch_full={_ms_batch_full:.1f}ms  '
                      f'prefix_fwd={_ms_pre:.1f}ms  '
                      f'batch_sfx_w_kv={_ms_batch_sfx:.1f}ms  '
                      f'net={_kv_net:+.1f}ms  '
                      f'{_breakeven_str}  '
                      f'→ {"BENEFICIAL" if _beneficial else "SKIP (batched-no-KV faster)"}')
            del _pkv_probe
            if self._dev.type == 'cuda': torch.cuda.empty_cache()

        _kv_beneficial = self._kv_probe_cache[_bucket]

        if not _kv_beneficial:
            self._root_pkv_obj = None
            return "", 0.0, prefix_len

        t0 = time.perf_counter()
        with torch.inference_mode():
            fwd = self.model(
                **prefix_inp,
                output_hidden_states = False,
                output_attentions    = False,
                use_cache            = True,
            )
        ms = _ms(t0)
        self._root_pkv_obj = fwd.past_key_values
        del fwd
        return "", ms, prefix_len

    def _get_ee_cfg(self) -> "Optional[tuple]":
        """Return early-exit config tuple for CGEE decode exit, else None."""
        if self.cfg.verify_with_cgee and getattr(
                self.cfg.rksc, 'cgee_early_exit_decode', True):
            return (self.cfg.rksc.cgee_min_decode_steps,
                    self.cfg.rksc.cgee_decode_conf_thresh,
                    self.cfg.rksc.cgee_decode_rel_gap)
        return None


    def _branch_passes(self, problem: str, root_text: str,
                       shared_prefix_len: int, depth: int) -> List[BranchResult]:
        """Generate all B branches — batched with ASKS KV sharing or batched no-KV."""
        B           = self.cfg.branching_factor
        prefix_text = self._shared_prefix(problem)
        dev         = self._dev
        pad_id      = self.tok.pad_token_id or self.tok.eos_token_id

        # Cache suffix token IDs — _branch_suffix is deterministic so we
        # only encode once per solver instance, saving B tokenizer calls/solve.
        if not hasattr(self, '_sfx_ids_cache'):
            self._sfx_ids_cache: Dict[int, torch.Tensor] = {}
        suffix_ids_list = []
        for b in range(B):
            if b not in self._sfx_ids_cache:
                self._sfx_ids_cache[b] = self.tok(
                    self._branch_suffix(b),
                    return_tensors='pt',
                    truncation=True,
                    max_length=128,
                )['input_ids']
            suffix_ids_list.append(self._sfx_ids_cache[b])

        max_sfx = max(s.shape[1] for s in suffix_ids_list)
        padded_sfx = torch.full((B, max_sfx), pad_id, dtype=torch.long, device=dev)
        sfx_attn   = torch.zeros(B, max_sfx, dtype=torch.long, device=dev)
        for i, ids in enumerate(suffix_ids_list):
            L = ids.shape[1]
            padded_sfx[i, max_sfx - L:] = ids[0].to(dev)
            sfx_attn[i, max_sfx - L:]   = 1

        _bucket_cur  = shared_prefix_len // 64
        _probe_cache = getattr(self, '_kv_probe_cache', {})
        _kv_ok_here  = _probe_cache.get(_bucket_cur, True)

        _tau = self.cfg.rksc.tau_similarity_threshold

        _kv_on = (
            not self.disable_reuse
            and _kv_ok_here
            and self._root_pkv_obj is not None
        )

        kv_reused = False
        gen_ids = gen_ms = _branch_conf = _exit_step = None

        if _kv_on:
            # ── ASKS path: expand root KV, decode all B branches batched ─────
            try:
                batch_kv  = self._expand_kv(self._root_pkv_obj, B)
                if batch_kv is not None:
                    prefix_ones     = torch.ones(B, shared_prefix_len,
                                                  dtype=torch.long, device=dev)
                    first_pass_mask = torch.cat([prefix_ones, sfx_attn], dim=1)
                    gen_ids, gen_ms, _branch_conf, _exit_step = self._stepwise_decode(
                        input_ids       = padded_sfx,
                        attention_mask  = first_pass_mask,
                        past_key_values = batch_kv,
                        max_new_tokens  = self.cfg.max_new_tokens_per_step,
                        early_exit_cfg  = self._get_ee_cfg(),
                    )
                    kv_reused = True
                    del batch_kv
            except Exception as _e:
                print(f'[ASKS] expand_kv failed: {_e} — falling back', flush=True)
                gen_ids = None

        if gen_ids is None:
            # ── Batched no-KV fallback ────────────────────────────────────────
            prefix_ids = self.tok(
                prefix_text, return_tensors='pt',
                truncation=True, max_length=self._max_seq_len,
            )['input_ids'].to(dev)
            max_full = max(prefix_ids.shape[1] + s.shape[1] for s in suffix_ids_list)
            batch_ids  = torch.full((B, max_full), pad_id, dtype=torch.long, device=dev)
            batch_mask = torch.zeros(B, max_full, dtype=torch.long, device=dev)
            for b, sfx_b in enumerate(suffix_ids_list):
                full_b = torch.cat([prefix_ids, sfx_b.to(dev)], dim=1)
                fl = full_b.shape[1]
                batch_ids[b, :fl]  = full_b[0]
                batch_mask[b, :fl] = 1
            gen_ids, gen_ms, _branch_conf, _exit_step = self._stepwise_decode(
                input_ids       = batch_ids,
                attention_mask  = batch_mask,
                past_key_values = None,
                max_new_tokens  = self.cfg.max_new_tokens_per_step,
                early_exit_cfg  = self._get_ee_cfg(),
            )
            kv_reused = False

        texts = [self.tok.decode(gen_ids[b], skip_special_tokens=True).strip()
                 for b in range(B)]
        try:
            _bconf = [float(_branch_conf[b]) for b in range(B)]
        except Exception as _bc_err:
            print(f'[V8] branch confidence fallback: {_bc_err}', flush=True)
            _bconf = [0.5] * B

        branches = []
        for b in range(B):
            sim = 1.0 if kv_reused else 0.0
            self.asks.records[b] = BranchSimilarityRecord(b, sim, kv_reused)
            importance = 1.0
            if self._root_layer_importances:
                li = min(depth - 1, len(self._root_layer_importances) - 1)
                importance = self._root_layer_importances[li]
            self.rsbcm.allocate(depth, b, importance=importance)
            branches.append(BranchResult(
                b, depth, texts[b], None,
                kv_reused, sim, None,
                gen_ms / B,
                generation_confidence = _bconf[b],
                decode_exit_step      = _exit_step,
                problem               = problem,
            ))
        return branches
    def _verify_cgee_batch(self, branches: List[BranchResult],
                           problem: str = '') -> List[BranchResult]:
        """
        Batched CGEE verify with true early exit via forward hooks.

        Registers a hook on each transformer layer that checks CGEE exit
        conditions for ALL branches in the batch.  When every branch has
        satisfied the entropy / stability gate, the hook raises
        _EarlyExitSignal to abort remaining layers — giving real compute
        savings while keeping full batching efficiency.
        """
        global _CGEE_EXIT_HISTORY

        _prob_ctx = problem[:2000] if problem else ''
        prompts = [
            f'Problem: {_prob_ctx}\n\n'
            f'Proposed answer: {b.text[:400]}\n\n'
            f'Is this answer correct? YES or NO.'
            for b in branches
        ]
        batch_enc = self.tok(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=self._max_seq_len,
        ).to(self._dev)
        padded   = batch_enc['input_ids']
        attn_msk = batch_enc['attention_mask'].long()
        lens     = attn_msk.sum(dim=-1).tolist()

        # ── CGEE config ──
        _base_theta = self.cfg.rksc.theta_entropy_threshold
        if len(_CGEE_EXIT_HISTORY) >= 20:
            _hist = np.array(_CGEE_EXIT_HISTORY[-100:], dtype=float)
            _scale = float(np.clip(
                1.0 - 0.1 * (_hist.mean() / max(self.arch.get('n_layers', 28), 1) - 0.5),
                0.85, 1.10))
            theta = _base_theta * _scale
        else:
            theta = _base_theta
        eps    = self.cfg.rksc.cgee_stability_eps
        min_l  = self.cfg.rksc.cgee_min_exit_layer
        unembed_f = self.arch['_unembed_weight'].float()
        B = len(branches)

        # ── Hook state for batch-level early exit ──
        _last_pos   = torch.tensor([l - 1 for l in lens], device=self._dev)
        _batch_arange = torch.arange(B, device=self._dev)
        _exited     = [False] * B          # per-branch exit flag
        _exit_layers = [None] * B          # per-branch exit layer
        _prev_ent   = [None] * B           # previous layer entropy
        _exit_hs    = [None] * B           # hidden state at exit layer
        _last_hs    = [None]               # most recent layer hidden state (mutable ref)
        _hooks      = []

        def _make_hook(layer_idx):
            def hook(module, inp, out):
                # out is (hidden_states, ...) or hidden_states tensor
                hs = out[0] if isinstance(out, tuple) else out
                # Extract last real token for each branch
                h_last = hs[_batch_arange, _last_pos, :].float().nan_to_num(0.0)
                _last_hs[0] = h_last  # always track latest

                if layer_idx < min_l:
                    return
                if all(_exited):
                    return  # already decided

                # Compute entropy for all branches at this layer
                logits = F.linear(h_last, unembed_f).clamp(-1e3, 1e3)
                probs  = logits.softmax(dim=-1)
                ent    = -(probs * (probs + 1e-10).log()).sum(dim=-1)  # [B]
                ent_vals = ent.cpu().tolist()
                del logits, probs, ent

                newly_exited = 0
                for i in range(B):
                    if _exited[i]:
                        continue
                    e = ent_vals[i]
                    if not np.isfinite(e):
                        _prev_ent[i] = None
                        continue
                    if (e < theta
                            and _prev_ent[i] is not None
                            and abs(e - _prev_ent[i]) < eps):
                        _exited[i] = True
                        _exit_layers[i] = layer_idx
                        _exit_hs[i] = h_last[i].clone()
                        newly_exited += 1
                    _prev_ent[i] = e

                if all(_exited):
                    raise _EarlyExitSignal(layer_idx)
            return hook

        # ── Register hooks ──
        layers = self._get_transformer_layers()
        for li, layer_module in enumerate(layers):
            _hooks.append(layer_module.register_forward_hook(_make_hook(li)))

        # ── Forward pass (may abort early) ──
        early_exited = False
        t0 = time.perf_counter()
        try:
            with torch.inference_mode():
                out = self.model(input_ids=padded, attention_mask=attn_msk,
                                 output_hidden_states=False, use_cache=False)
            # Full depth reached — grab final hidden states for non-exited branches
            final_hs = _last_hs[0]
        except _EarlyExitSignal as sig:
            early_exited = True
            final_hs = _last_hs[0]
            out = None
        ms_batch = _ms(t0)

        # ── Clean up hooks ──
        for h in _hooks:
            h.remove()

        # ── Score branches ──
        _yes_id, _no_id = self._yes_id, self._no_id
        for i, branch in enumerate(branches):
            if _exit_layers[i] is not None:
                h = _exit_hs[i]
                branch.cgee_exit_layer = _exit_layers[i]
            else:
                h = final_hs[i] if final_hs is not None else None
                branch.cgee_exit_layer = len(layers) - 1

            branch.latency_ms += ms_batch / B

            if h is not None and _yes_id is not None and _no_id is not None:
                logits_yn = F.linear(h.unsqueeze(0), unembed_f).clamp(-1e3, 1e3)
                logit_yes = logits_yn[0, _yes_id].item()
                logit_no  = logits_yn[0, _no_id].item()
                branch.score = float(
                    torch.tensor([logit_yes, logit_no]).softmax(0)[0].item())
            else:
                branch.score = branch.generation_confidence

            if _exit_layers[i] is not None:
                _CGEE_EXIT_HISTORY.append(_exit_layers[i])

        del _exit_hs, _last_hs, final_hs
        if out is not None:
            del out
        return branches


    def _verify_smart(self, branches: List[BranchResult],
                      problem: str = '') -> List[BranchResult]:
        """
        Verify-pass CGEE: skip the verify forward pass when generation is decisive.
        Complements decode-level CGEE (which saves decode steps).
        Two conditions (either triggers skip):
          A. top_conf >= thresh AND abs_gap >= gap
          B. top_conf >= 0.9×thresh AND rel_gap >= rel_thresh
        """
        if not hasattr(self, '_cgee_skip_count'):
            self._cgee_skip_count = 0; self._cgee_call_count = 0
        self._cgee_call_count += 1

        if len(branches) < 2:
            branches[0].score = branches[0].generation_confidence
            self._cgee_skip_count += 1
            return branches

        confs    = [b.generation_confidence for b in branches]
        sc       = sorted(confs, reverse=True)
        top, sec = sc[0], sc[1]
        abs_gap  = top - sec
        rel_gap  = abs_gap / max(top, 1e-6)

        thresh    = self.cfg.rksc.cgee_gen_conf_threshold
        gap       = self.cfg.rksc.cgee_gen_conf_gap
        use_rel   = getattr(self.cfg.rksc, 'cgee_use_relative_gap', True)
        rel_t     = getattr(self.cfg.rksc, 'cgee_relative_gap_thresh', 0.08)

        skip = (top >= thresh and abs_gap >= gap)
        if use_rel and not skip:
            skip = (top >= thresh * 0.90 and rel_gap >= rel_t)

        if skip:
            for b in branches: b.score = b.generation_confidence
            self._cgee_skip_count += 1
            return branches
        return self._verify_cgee_batch(branches, problem=problem)

    def _verify_batch_full(self, branches: List[BranchResult],
                           problem: str = '') -> List[BranchResult]:
        """
        Batched full-depth verify scored by YES/NO next-token probability.
        Includes problem context so the model evaluates correctness, not fluency.
        Passes output_hidden_states=True so CGEEAnalyzer can record the exit
        layer that would have fired (for measurement), even though the batched
        forward cannot physically abort early.  True compute savings from CGEE
        require the serial path in _verify_smart.
        """
        _prob_ctx = problem[:2000] if problem else ''
        prompts  = [
            f'Problem: {_prob_ctx}\n\nProposed answer: {b.text[:400]}\n\n'
            f'Is this answer correct? YES or NO.'
            for b in branches
        ]
        batch_enc = self.tok(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=self._max_seq_len,
        ).to(self._dev)
        padded   = batch_enc['input_ids']
        attn_msk = batch_enc['attention_mask'].long()
        lens     = attn_msk.sum(dim=-1).tolist()

        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.model(input_ids=padded, attention_mask=attn_msk,
                             output_hidden_states=True, use_cache=False)
        ms_batch = _ms(t0)

        _yes_id, _no_id = self._yes_id, self._no_id
        _tok_idx   = torch.tensor([l-1 for l in lens], device=self._dev)
        _batch_idx = torch.arange(len(branches), device=self._dev)
        last_logits = out.logits[_batch_idx, _tok_idx, :].float()

        # CGEE exit-layer analysis per branch (measurement only in batched mode).
        if out.hidden_states is not None:
            for i, branch in enumerate(branches):
                last_pos  = lens[i] - 1
                branch_hs = tuple(
                    h[i:i+1, last_pos:last_pos+1, :]
                    for h in out.hidden_states
                )
                result = self.cgee.analyze(_HSWrapper(branch_hs))
                branch.cgee_exit_layer = result.exit_layer
        del out

        for i, branch in enumerate(branches):
            if _yes_id is not None and _no_id is not None:
                logit_yes = last_logits[i, _yes_id].item()
                logit_no  = last_logits[i, _no_id].item()
                branch.score = float(torch.tensor([logit_yes, logit_no]).softmax(0)[0].item())
            else:
                p = last_logits[i].softmax(dim=-1)
                branch.score = float(p.max().item())
            branch.latency_ms += ms_batch / len(branches)
        del last_logits
        return branches


    def solve(self, problem: str) -> SolveResult:
        self.asks.reset()
        self.rsbcm.reset()
        t_total = time.perf_counter()
        self._n_solved += 1

        root_text, root_ms, shared_prefix_len = self._root_pass(problem)

        branches = self._branch_passes(problem, root_text, shared_prefix_len, depth=1)

        if self.cfg.verify_with_cgee:
            branches = self._verify_smart(branches, problem=problem)
        else:
            branches = self._verify_batch_full(branches, problem=problem)

        # Composite selection: verify score weighted by generation confidence.
        # sqrt dampens confidence so a 2× confidence advantage only adds ~41%
        # weight — avoids over-trusting high-confidence wrong answers.
        def _composite(b):
            s = b.score if b.score is not None else 0.5
            c = b.generation_confidence if b.generation_confidence is not None else 0.5
            return s * (c ** 0.5)
        best = max(branches, key=_composite)

        if not hasattr(self, '_decode_exit_history'):
            self._decode_exit_history: list = []
        _es = branches[0].decode_exit_step if branches else -1
        self._decode_exit_history.append(_es)

        self._root_pkv_obj = None

        return SolveResult(
            problem            = problem,
            answer             = best.text,
            branches           = branches,
            root_latency_ms    = root_ms,
            total_latency_ms   = _ms(t_total),
            nowaste_latency_ms = 0.0,
        )


RKSCSolver = RKSCSolverV8
print('RKSCSolver ready.')


## §6  Table 1 — Latency Decomposition

Three conditions, all measured vs a **Batched-no-KV** reference (B branches batched, no prefix KV sharing):

| Condition | What it isolates |
|---|---|
| **KV only** (RKSC ASKS, no verify skip) | Pure KV-sharing gain |
| **CGEE only** (no KV, verify-skip active) | Pure verify-skip gain |
| **RKSC** (KV + CGEE combined) | End-to-end throughput gain |

Problems are padded to a **1024-token shared prefix** — at this length, prefix attention (O(n²)) dominates and KV sharing saves one full prefill per problem across all B branches.


In [ ]:
PREFIX_TARGET = 1024

print(f'Padding {len(gpqa_probs)} problems to ~{PREFIX_TARGET}-token shared prefix ...')
probs_padded = [pad_to_target(p, tokenizer, PREFIX_TARGET) for p in gpqa_probs]

_prefix_lens = [
    int(tokenizer(_PAD_TEMPLATE.format(q=p), return_tensors='pt',
                  truncation=True, max_length=MAX_SEQ_LEN
                  )['input_ids'].shape[1])
    for p in probs_padded
]
print(f'Shared prefix — mean: {np.mean(_prefix_lens):.0f} tok  '
      f'min: {min(_prefix_lens)}  max: {max(_prefix_lens)}')
print(f'At prefix≈{np.mean(_prefix_lens):.0f} tokens with B={B_FACTOR}, '
      'KV caching should be beneficial.')


In [ ]:
def benchmark_condition(model, tokenizer, problems, cfg, arch,
                         disable_reuse, n_runs=2, label='',
                         full_depth_verify=False, batch_no_kv=False,
                         n_warmup=0):
    """Run a single condition across `problems`.

    `n_warmup` solves are executed on the first problem before timing
    begins.  The first few RKSC solves pay a CUDA graph / allocator
    warmup cost and inflate variance; running a handful of discarded
    solves brings the RKSC CV down from ~30% to ~10% without changing
    the mean.
    """
    cfg2                  = deepcopy(cfg)
    cfg2.batch_without_kv = batch_no_kv
    if disable_reuse:
        cfg2.rksc.tau_similarity_threshold = 1.1
        cfg2.verify_full_depth             = full_depth_verify
        cfg2.verify_with_cgee              = False
    else:
        cfg2.verify_full_depth             = full_depth_verify

    solver = RKSCSolverV8(model, tokenizer, cfg2, arch, disable_reuse=disable_reuse)

    if n_warmup > 0 and problems:
        for _ in range(n_warmup):
            _warm = solver.solve(problems[0])
            del _warm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    lats, reuse_rates = [], []
    last = None
    for prob in tqdm(problems, desc=label or f'disable={disable_reuse}'):
        run_lats = []
        for _ in range(n_runs):
            res = solver.solve(prob)
            run_lats.append(res.total_latency_ms)
            last = res
        lats.append(float(np.mean(run_lats)))
        if last:
            reuse_rates.append(last.kv_reuse_rate)
        if len(lats) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    solver.remove_hooks()
    del solver, last
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return lats, reuse_rates


In [ ]:
cal    = CALIBRATED.get(MODEL_ID, {'tau': 0.75, 'theta': 12.0})
cfg_bm = RKSCSolverConfig(
    branching_factor        = B_FACTOR,
    max_depth               = MAX_DEPTH,
    max_new_tokens_per_step = MAX_NEW_TOK,
)
cfg_bm.rksc.tau_similarity_threshold = cal['tau']
cfg_bm.rksc.theta_entropy_threshold  = cal['theta']
cfg_bm.rksc.prism_enabled            = False


def _gpu_free_gb() -> float:
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
        return torch.cuda.mem_get_info()[0] / 1e9
    return 0.0


def _run_condition(label, ckpt_key, cfg_override=None,
                   disable=False, full_depth=False, batch_no_kv=False):
    _vkey  = f'{ckpt_key}_{CKPT_VERSION}'.replace('/', '_').replace('|', '_')
    cached = ckpt_load(_vkey)
    if cached is not None:
        print(f'{label}: loaded from cache', flush=True)
        return cached['lats'], cached['rr']
    cfg_use = deepcopy(cfg_override or cfg_bm)
    print(f'{label}: running ... (VRAM {_gpu_free_gb():.1f} GB free)', flush=True)
    lats, rr = benchmark_condition(
        model, tokenizer, probs_padded, cfg_use, ARCH,
        disable_reuse=disable, n_runs=N_RUNS,
        full_depth_verify=full_depth, batch_no_kv=batch_no_kv,
        label=label, n_warmup=N_WARMUP)
    ckpt_save(_vkey, {'lats': lats, 'rr': rr})
    print(f'  Done. mean={np.mean(lats):.1f} ms  CV={cv_percent(lats):.0f}%')
    return lats, rr


lat_batch_nokv, _ = _run_condition(
    'Batched-no-KV (reference)', 'lat_batch_nokv',
    disable=True, full_depth=True, batch_no_kv=True)

cfg_kv = deepcopy(cfg_bm)
cfg_kv.verify_with_cgee  = False
cfg_kv.verify_full_depth = True
lat_kv, rr_kv = _run_condition('KV only', 'lat_kv', cfg_kv)

cfg_cgee = deepcopy(cfg_bm)
cfg_cgee.verify_with_cgee  = True
cfg_cgee.verify_full_depth = False
lat_cgee, rr_cgee = _run_condition(
    'CGEE only', 'lat_cgee', cfg_cgee, disable=True, batch_no_kv=True)

lat_rksc, rr_rksc = _run_condition('RKSC (KV + CGEE)', 'lat_rksc')


In [ ]:
def speedup(sys_lats, base_lats):
    """Ratio-of-means speedup and percent saved."""
    sp  = float(np.mean(base_lats)) / float(np.mean(sys_lats))
    pct = (1.0 - 1.0 / sp) * 100.0
    return sp, pct


_, p_kv   = paired_ttest(lat_kv,   lat_batch_nokv)
_, p_cgee = paired_ttest(lat_cgee, lat_batch_nokv)
_, p_rksc = paired_ttest(lat_rksc, lat_batch_nokv)

ref_ms = float(np.mean(lat_batch_nokv))

ci_kv   = bootstrap_ci_ratio(lat_batch_nokv, lat_kv)
ci_cgee = bootstrap_ci_ratio(lat_batch_nokv, lat_cgee)
ci_rksc = bootstrap_ci_ratio(lat_batch_nokv, lat_rksc)

cv_batch = cv_percent(lat_batch_nokv)
cv_kv    = cv_percent(lat_kv)
cv_cgee  = cv_percent(lat_cgee)
cv_rksc  = cv_percent(lat_rksc)

rows = []
for label, lats, rr, p in [
    ('Batched-no-KV (reference)', lat_batch_nokv, [0.0]*len(lat_batch_nokv), None),
    ('KV only',                   lat_kv,         rr_kv,                    p_kv),
    ('CGEE only',                 lat_cgee,       rr_cgee,                  p_cgee),
    ('RKSC (KV + CGEE)',          lat_rksc,       rr_rksc,                  p_rksc),
]:
    sp, pct = speedup(lats, lat_batch_nokv)
    rows.append({
        'Condition': label,
        'Mean (ms)': f'{np.mean(lats):.1f}±{np.std(lats):.1f}',
        'Speedup':   f'{sp:.3f}×' if sp > 1.001 else '1.000× (ref)',
        '% saved':   f'{pct:.1f}%' if sp > 1.001 else '—',
        'KV reuse':  f'{np.mean(rr):.0%}' if any(r > 0 for r in rr) else '—',
        'p-value':   f'{p:.4f}' if p is not None else '—',
    })

df_t1 = pd.DataFrame(rows)

print()
print('=' * 75)
print('TABLE 1 — Latency Decomposition')
print(f'Model: {MODEL_ID}   B={B_FACTOR}  tok={MAX_NEW_TOK}  '
      f'N={N_PROBLEMS}  prefix≈{PREFIX_TARGET}')
print('Reference: Batched-no-KV  |  all speedups measured against this baseline')
print('=' * 75)
print(df_t1.to_string(index=False))
print()

sp_kv,   pct_kv   = speedup(lat_kv,   lat_batch_nokv)
sp_cgee, pct_cgee = speedup(lat_cgee, lat_batch_nokv)
sp_rksc, pct_rksc = speedup(lat_rksc, lat_batch_nokv)

print('Speedup decomposition (bootstrap 95%% CI, N={0}, n_runs={1}):'.format(
    N_PROBLEMS, N_RUNS))
print(f'  KV sharing alone  : {sp_kv:.3f}×  '
      f'[95%CI {ci_kv[0]:.3f}–{ci_kv[1]:.3f}]  '
      f'({ref_ms - np.mean(lat_kv):.0f} ms saved, {pct_kv:.1f}%)  '
      f'CV={cv_kv:.0f}%  p={p_kv:.4f}')
print(f'  CGEE alone        : {sp_cgee:.3f}×  '
      f'[95%CI {ci_cgee[0]:.3f}–{ci_cgee[1]:.3f}]  '
      f'({ref_ms - np.mean(lat_cgee):.0f} ms saved, {pct_cgee:.1f}%)  '
      f'CV={cv_cgee:.0f}%  p={p_cgee:.4f}')
print(f'  RKSC combined     : {sp_rksc:.3f}×  '
      f'[95%CI {ci_rksc[0]:.3f}–{ci_rksc[1]:.3f}]  '
      f'({ref_ms - np.mean(lat_rksc):.0f} ms saved, {pct_rksc:.1f}%)  '
      f'CV={cv_rksc:.0f}%  p={p_rksc:.4f}')
print(f'  Baseline CV       : {cv_batch:.0f}%  N={N_PROBLEMS}  n_runs={N_RUNS}')

# Additivity: gap with CI honesty rather than headlining "super-additive".
save_kv   = ref_ms - float(np.mean(lat_kv))
save_cgee = ref_ms - float(np.mean(lat_cgee))
save_rksc = ref_ms - float(np.mean(lat_rksc))
gap_ms    = save_rksc - (save_kv + save_cgee)
# The CGEE-alone CI already brackets zero → gap magnitude is noise-dominated
# whenever CGEE's individual contribution is not separable from baseline CV.
cgee_separable = (ci_cgee[0] > 1.000) and (cv_cgee < cv_batch * 3)
print()
print('Additivity check:')
print(f'  KV saved   : {save_kv:.0f} ms     CGEE saved : {save_cgee:.0f} ms')
print(f'  RKSC saved : {save_rksc:.0f} ms   gap = {gap_ms:+.0f} ms')
if not cgee_separable:
    print('  NOTE: CGEE contribution at MAX_NEW_TOK={0} is not separable from'.format(
        MAX_NEW_TOK))
    print('        baseline noise — apparent gap is variance, not synergy.')
    print('        See §6b (verify-only) for an isolated CGEE measurement.')
else:
    label = 'sub-additive' if gap_ms < 0 else 'super-additive'
    print(f'  Components combine {label} ({gap_ms:+.0f} ms beyond sum).')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ref      = float(np.mean(lat_batch_nokv))
conds    = ['Batched\nno-KV',
            f'KV only\n(τ={cfg_bm.rksc.tau_similarity_threshold})',
            f'CGEE only\n(θ={cfg_bm.rksc.theta_entropy_threshold})',
            f'RKSC\n(KV+CGEE)']
all_lats = [lat_batch_nokv, lat_kv, lat_cgee, lat_rksc]
colors   = ['#7f7f7f', '#1f77b4', '#ff7f0e', '#2ca02c']

# ── Left: latency distribution (boxplot) ─────────────────────────────
ax1 = axes[0]
bp  = ax1.boxplot(all_lats, patch_artist=True, widths=0.55,
                   medianprops={'color': '#222', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax1.set_xticks(range(1, len(conds) + 1))
ax1.set_xticklabels(conds, fontsize=9)
ax1.set_ylabel('Per-problem latency (ms)')
ax1.set_title(f'Latency by condition ({MODEL_ID.split("/")[-1]})')
ax1.axhline(ref, color='#7f7f7f', linestyle=':', linewidth=1)
ax1.grid(axis='y', alpha=0.3)

# ── Right: absolute ms saved with 95% bootstrap bars ──────────────
ax2  = axes[1]
save = [ref - float(np.mean(x)) for x in (lat_kv, lat_cgee, lat_rksc)]
cis  = [ci_kv, ci_cgee, ci_rksc]
# Convert speedup CI → ms-saved CI (approximate, one-sided sanity interval)
err  = [[ref - ref / lo - s, ref - ref / hi + s][0]  # not used — keep simple
         for (lo, hi), s in zip(cis, save)]
bars = ax2.bar(conds[1:], save, color=colors[1:],
                alpha=0.85, edgecolor='white', linewidth=0.8)
pcts = [(s / ref) * 100 for s in save]
for bar, s, pct in zip(bars, save, pcts):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + max(save) * 0.02,
             f'{s:+.0f} ms\n({pct:+.1f}%)',
             ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Latency saved vs baseline (ms)')
ax2.set_title('Absolute savings per condition')
ax2.set_ylim(min(0, min(save) * 1.2), max(save) * 1.35)
ax2.axhline(0, color='#333', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Table 1 — Latency decomposition (RKSC vs individual components)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_table1_decomposition.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_table1_decomposition.pdf')


## §6b  CGEE verify-only isolation

Table 1's CGEE-only row (0.6%) understates CGEE's contribution because the
short `MAX_NEW_TOK=32` regime makes the prefill phase dominate total latency
by ~10× — early exit during decode cannot save what decode does not spend.
This section re-times only the verify pass with a longer decode budget so
the CGEE contribution is resolvable above baseline noise.


In [ ]:
from copy import deepcopy as _dc

N_VERIFY          = 15
VERIFY_DECODE_TOK = MAX_NEW_TOK * 4


def time_verify_condition(use_cgee: bool, label: str):
    cfg = _dc(cfg_bm)
    cfg.max_new_tokens_per_step = VERIFY_DECODE_TOK
    cfg.verify_with_cgee  = False
    cfg.verify_full_depth = True
    solver = RKSCSolverV8(model, tokenizer, cfg, ARCH, disable_reuse=False)

    method = '_verify_cgee_batch' if use_cgee else '_verify_batch_full'
    orig   = getattr(solver, method)
    times: list = []

    def timed(*args, **kwargs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = orig(*args, **kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1e3)
        return out

    setattr(solver, '_verify_batch_full', timed)
    solver.solve(probs_padded[0])
    times.clear()

    exit_layers: list = []
    for prob in tqdm(probs_padded[:N_VERIFY], desc=label):
        res = solver.solve(prob)
        exit_layers += [b.cgee_exit_layer for b in res.branches
                        if b.cgee_exit_layer is not None]
        del res

    solver.remove_hooks()
    del solver
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return times, exit_layers


print(f'\n§6b  CGEE verify-only isolation  (N={N_VERIFY}, verify_decode={VERIFY_DECODE_TOK})')
lat_ver_full, _         = time_verify_condition(False, 'verify full-depth')
lat_ver_cgee, exit_lyrs = time_verify_condition(True,  'verify CGEE (batched early-exit)')

if not lat_ver_full or not lat_ver_cgee:
    print('  WARNING: verify phase not invoked — check solver path.')
    sp_ver = ci_ver = p_ver = saved_ver = float('nan')
else:
    n      = min(len(lat_ver_full), len(lat_ver_cgee))
    sp_ver    = float(np.mean(lat_ver_full)) / float(np.mean(lat_ver_cgee))
    ci_ver    = bootstrap_ci_ratio(lat_ver_full, lat_ver_cgee)
    _, p_ver  = paired_ttest(lat_ver_full[:n], lat_ver_cgee[:n])
    saved_ver = float(np.mean(lat_ver_full)) - float(np.mean(lat_ver_cgee))

n_layers      = ARCH.get('n_layers', 32)
fire_rate     = len(exit_lyrs) / (N_VERIFY * cfg_bm.branching_factor) if exit_lyrs else 0
mean_exit_lyr = float(np.mean(exit_lyrs)) if exit_lyrs else float('nan')

print()
print('Verify-only latency:')
if lat_ver_full and lat_ver_cgee:
    print(f'  Full-depth : {np.mean(lat_ver_full):7.1f} ± {np.std(lat_ver_full):.1f} ms  (CV={cv_percent(lat_ver_full):.0f}%)')
    print(f'  CGEE       : {np.mean(lat_ver_cgee):7.1f} ± {np.std(lat_ver_cgee):.1f} ms  (CV={cv_percent(lat_ver_cgee):.0f}%)')
    print(f'  Speedup    : {sp_ver:.3f}×  [95%CI {ci_ver[0]:.3f}–{ci_ver[1]:.3f}]  p={p_ver:.4f}  (saved {saved_ver:.0f} ms)')
    print(f'  Fire rate  : {fire_rate:.1%}  (mean exit layer {mean_exit_lyr:.1f} / {n_layers})')
    print()
    if fire_rate > 0:
        print(f'  CGEE exited early on {fire_rate:.1%} of branches (mean layer {mean_exit_lyr:.1f}/{n_layers}).')
    else:
        print('  NOTE: CGEE gate did not fire — lower theta_entropy_threshold or cgee_min_exit_layer.')
else:
    print('  (no measurements — verify skipped)')


## §7  Extended Evaluation — Multi-Model, Multi-Dataset

Tests across 7B–14B GQA models and four datasets spanning difficulty tiers:

| Dataset | Difficulty | Expected CGEE skip rate |
|---|---|---|
| GSM8K | Easy (math) | 60–80% |
| ARC-Challenge | Medium | 40–60% |
| MMLU-STEM | Medium-hard | 20–40% |
| GPQA Diamond | Hard (grad-level) | 10–30% |


In [ ]:
EVAL_MODELS_PRIORITY = [
    # 'Qwen/Qwen2.5-7B-Instruct',
    # 'mistralai/Mistral-7B-Instruct-v0.3',
    # 'tiiuae/Falcon3-7B-Instruct',
    # 'tiiuae/Falcon3-10B-Instruct',
    'meta-llama/Meta-Llama-3-8B-Instruct',
]

if torch.cuda.is_available():
    _vram_free = torch.cuda.mem_get_info()[0] / 1e9
    EVAL_MODELS = [m for m in EVAL_MODELS_PRIORITY
                   if MODEL_VRAM_GB.get(m, 30.0) + 2.0 <= _vram_free]
    print(f'VRAM available : {_vram_free:.1f} GB')
    print(f'Models selected: {len(EVAL_MODELS)}')
    for m in EVAL_MODELS:
        print(f'  {m}  ({MODEL_VRAM_GB.get(m, 0):.0f} GB)')
else:
    EVAL_MODELS = []
    print('No CUDA — extended eval requires GPU')

_GPQA_PREAMBLE_EXT = (
    'You are solving a graduate-level science problem. '
    'Reason carefully and show all steps.\n\n'
)

EVAL_DATASETS = {
    'gpqa': {
        'hf_name': 'Idavidrein/gpqa', 'config': 'gpqa_diamond', 'split': 'train',
        'q_field': 'Question', 'a_field': 'Correct Answer',
        'preamble': _GPQA_PREAMBLE_EXT,
    },
    'gsm8k': {
        'hf_name': 'openai/gsm8k', 'config': 'main', 'split': 'test',
        'q_field': 'question', 'a_field': 'answer',
        'preamble': 'Solve the following math problem step by step.\n\n',
    },
    'arc_challenge': {
        'hf_name': 'allenai/ai2_arc', 'config': 'ARC-Challenge', 'split': 'test',
        'q_field': 'question', 'a_field': 'answerKey',
        'preamble': 'Answer this multiple-choice science question. Think step by step.\n\n',
    },
    'mmlu_stem': {
        'hf_name': 'cais/mmlu', 'config': 'college_physics', 'split': 'test',
        'q_field': 'question', 'a_field': 'answer',
        'preamble': 'Answer this university-level physics question.\n\n',
    },
}

N_EVAL      = 50
N_EVAL_RUNS = 2
print(f'\nN_EVAL={N_EVAL}  tok={MAX_NEW_TOK_EXT}  B={B_FACTOR}')
_est = len(EVAL_MODELS) * len(EVAL_DATASETS) * N_EVAL * N_EVAL_RUNS * 4 * 3 / 60
print(f'Est. runtime: ~{_est:.0f} min  (4 conditions × {len(EVAL_MODELS)} models × {len(EVAL_DATASETS)} datasets × {N_EVAL_RUNS} runs)')


In [ ]:
def load_eval_dataset(ds_name: str, ds_cfg: Dict, n: int):
    """Returns (questions: List[str], ground_truth: List[str]).
    Errors propagate to the caller, which handles per-dataset fallback."""
    preamble = ds_cfg.get('preamble', '')
    kw = dict(split=ds_cfg['split'])
    if ds_cfg.get('config'):
        kw['name'] = ds_cfg['config']
    ds = load_dataset(ds_cfg['hf_name'], **kw)
    ds = ds.select(range(min(n, len(ds))))
    probs   = [preamble + str(ex[ds_cfg['q_field']]) for ex in ds]
    a_field = ds_cfg.get('a_field')
    ans     = [str(ex[a_field]) if a_field and a_field in ex else '' for ex in ds]
    print(f'  {ds_name}: loaded {len(probs)} problems')
    return probs, ans

print('Loading extended eval datasets ...')
EVAL_DATA:       Dict[str, List[str]] = {}
EVAL_GT_ANSWERS: Dict[str, List[str]] = {}   # ground-truth, keyed by dataset name
for ds_name, ds_cfg in EVAL_DATASETS.items():
    try:
        if ds_name == 'gpqa' and 'gpqa_probs' in globals() and gpqa_probs:
            EVAL_DATA[ds_name]       = gpqa_probs[:N_EVAL]
            EVAL_GT_ANSWERS[ds_name] = (gpqa_ans[:N_EVAL]
                                         if 'gpqa_ans' in globals() else [])
            print(f'  gpqa: reusing {len(EVAL_DATA[ds_name])} pre-loaded problems')
        else:
            _qs, _as = load_eval_dataset(ds_name, ds_cfg, N_EVAL)
            EVAL_DATA[ds_name]       = _qs
            EVAL_GT_ANSWERS[ds_name] = _as
        avg_tok = np.mean(tokenizer(EVAL_DATA[ds_name], truncation=True,
                                   max_length=MAX_SEQ_LEN,
                                   return_length=True)['length'])
        print(f'  {ds_name}: {len(EVAL_DATA[ds_name])} problems, '
              f'avg_tokens={avg_tok:.0f}')
    except Exception as _e:
        print(f'  {ds_name}: SKIPPED — {type(_e).__name__}: {_e}')
        EVAL_DATA.pop(ds_name, None); EVAL_GT_ANSWERS.pop(ds_name, None)

if not EVAL_DATA:
    print('⚠  No datasets loaded successfully — extended eval will be empty.')


In [ ]:
import re

def _extract_gsm8k_number(gt: str) -> str:
    m = re.search(r'####\s*([\d,\.\-]+)', gt)
    return m.group(1).replace(',', '').strip() if m else gt.strip()

def _extract_mmlu_letter(gt_raw: str, choices) -> str:
    letters = ['A', 'B', 'C', 'D']
    try:
        idx = int(gt_raw)
        choice_text = choices[idx] if choices and idx < len(choices) else ''
        return f'{letters[idx]}. {choice_text}'.strip()
    except (ValueError, IndexError, TypeError):
        return str(gt_raw).strip()

def _acc_vs_gt(answers, gt_list, ds_name, choices_list=None):
    """Format-aware accuracy: GSM8K number match, ARC/MMLU letter match, else substring."""
    if not gt_list:
        return float('nan'), []
    hits = []
    for i, (ans, gt) in enumerate(zip(answers, gt_list)):
        ans_s, gt_s = ans.strip(), str(gt).strip()
        if ds_name == 'gsm8k':
            gt_num = _extract_gsm8k_number(gt_s)
            nums   = re.findall(r'[\-]?\d+(?:[,\.]\d+)*', ans_s)
            pred   = nums[-1].replace(',', '') if nums else ''
            try:    hit = float(pred) == float(gt_num)
            except ValueError: hit = pred == gt_num
        elif ds_name in ('arc_challenge', 'mmlu_stem'):
            choices = choices_list[i] if choices_list and i < len(choices_list) else None
            gt_norm = (_extract_mmlu_letter(gt_s, choices)
                       if ds_name == 'mmlu_stem' else gt_s.upper().strip())
            letter  = gt_norm[0] if gt_norm else ''
            hit     = bool(letter and re.search(rf'\b{re.escape(letter)}\b', ans_s, re.IGNORECASE))
        else:
            gt_l = gt_s.lower()
            hit  = bool(gt_l and (gt_l in ans_s.lower() or ans_s.lower() in gt_l))
        hits.append(hit)
    return sum(hits) / max(len(hits), 1), hits


def _pad_for_extended_eval(problem: str, tokenizer, target: int = 1024,
                            max_seq_len: int = 2048) -> str:
    return pad_to_target(problem, tokenizer, target, max_seq_len)


def run_extended_benchmark(
    model_id: str,
    datasets: Dict[str, List[str]],
    n_runs: int = 1,
    gt_answers: Optional[Dict[str, List[str]]] = None,
) -> Dict:
    """
    Three-condition benchmark across all datasets for a single model.

    Conditions: NoKV (reference) | vLLM-equiv (τ=0, always-reuse) | RKSC+CGEE (τ=calibrated).
    Full-verify latency is captured inside the RKSC+CGEE solve loop to avoid
    a redundant fourth pass.
    """
    print(f'\n{"="*60}\nModel: {model_id}\n{"="*60}')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        needed  = MODEL_VRAM_GB.get(model_id, 30.0)
        if free_gb < needed + 2.0:
            print(f'  SKIPPED: {free_gb:.1f} GB free, need {needed:.0f}+2 GB')
            return {}

    global _KV_PROBE_VERBOSE
    _KV_PROBE_VERBOSE = False

    model, tok = load_model_and_tokenizer(model_id, attn_impl=ATTN_IMPL)
    arch = diagnose_architecture(model, tok)
    arch['_unembed_weight'] = model.get_output_embeddings().weight.data
    cal = CALIBRATED.get(model_id, {'tau': 0.82, 'theta': 6.0})

    n_layers = arch.get('n_layers', '?')
    n_kv     = arch.get('num_key_value_heads', '?')
    n_q      = arch.get('num_attention_heads', '?')
    head_dim = arch.get('head_dim', '?')
    print(f'  Arch: {n_layers}L  {n_q}Q/{n_kv}KV  head_dim={head_dim}')
    try:
        kv_mb = 2 * int(n_layers) * int(n_kv) * int(head_dim) * 1024 * 2 / 1e6
        print(f'  KV/branch @ prefix=1024: {kv_mb:.0f} MB  '              f'(sharing saves {(B_FACTOR-1)*kv_mb:.0f} MB vs B={B_FACTOR} caches)')
    except (ValueError, TypeError):
        pass

    results = {}

    for ds_name, problems in datasets.items():
        print(f'\n  Dataset: {ds_name} ({len(problems)} problems)')

        padded = [pad_to_target(p, tok, target=1024, max_seq_len=MAX_SEQ_LEN) for p in problems]
        avg_raw = float(np.mean(tok(problems, truncation=True, max_length=MAX_SEQ_LEN, return_length=True)['length']))
        avg_pad = float(np.mean(tok(padded,   truncation=True, max_length=MAX_SEQ_LEN, return_length=True)['length']))
        print(f'    prefix tokens: raw={avg_raw:.0f}  padded={avg_pad:.0f}')

        base_cfg = RKSCSolverConfig(
            branching_factor=B_FACTOR, max_depth=MAX_DEPTH,
            max_new_tokens_per_step=MAX_NEW_TOK_EXT, max_seq_len=MAX_SEQ_LEN)
        base_cfg.rksc.tau_similarity_threshold = cal['tau']
        base_cfg.rksc.theta_entropy_threshold  = cal['theta']
        base_cfg.rksc.prism_enabled            = False

        def run_condition(label, tau, disable_reuse, batch_no_kv):
            cfg = deepcopy(base_cfg)
            cfg.rksc.tau_similarity_threshold = tau
            cfg.verify_with_cgee  = False
            cfg.verify_full_depth = False
            return benchmark_condition(
                model, tok, padded, cfg, arch,
                disable_reuse=disable_reuse, batch_no_kv=batch_no_kv,
                full_depth_verify=False, n_runs=n_runs, n_warmup=1,
                label=f'  {ds_name} {label}')

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            lat_nokv, _ = run_condition('NoKV', tau=1.1, disable_reuse=True, batch_no_kv=True)
            peak_nokv_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0.0

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            lat_vllm, _ = run_condition('vLLM-equiv', tau=0.0, disable_reuse=False, batch_no_kv=False)
            peak_rksc_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0.0
            mem_saved_mb = peak_nokv_mb - peak_rksc_mb

            cfg_cgee = deepcopy(base_cfg)
            cfg_cgee.verify_with_cgee  = True
            cfg_cgee.verify_full_depth = False

            cfg_verify = deepcopy(base_cfg)
            cfg_verify.verify_with_cgee  = False
            cfg_verify.verify_full_depth = True

            solver_cgee   = RKSCSolverV8(model, tok, cfg_cgee,   arch, disable_reuse=False)
            solver_verify = RKSCSolverV8(model, tok, cfg_verify, arch, disable_reuse=False)
            solver_cgee.solve(padded[0])
            solver_verify.solve(padded[0])

            lat_cgee, lat_verify_inner = [], []
            ans_cgee, ans_verify       = [], []

            for prob in tqdm(padded, desc=f'  {ds_name} RKSC+CGEE', leave=False):
                cgee_runs, verify_runs = [], []
                for _ in range(n_runs):
                    r_cgee   = solver_cgee.solve(prob)
                    r_verify = solver_verify.solve(prob)
                    cgee_runs.append(r_cgee.total_latency_ms)
                    verify_runs.append(r_verify.total_latency_ms)
                lat_cgee.append(float(np.mean(cgee_runs)))
                lat_verify_inner.append(float(np.mean(verify_runs)))
                ans_cgee.append(getattr(r_cgee,   'answer', '') or '')
                ans_verify.append(getattr(r_verify, 'answer', '') or '')

            cgee_skip_n    = getattr(solver_cgee, '_cgee_skip_count', 0)
            cgee_call_n    = getattr(solver_cgee, '_cgee_call_count', 1)
            cgee_skip_rate = cgee_skip_n / max(cgee_call_n, 1)

            solver_cgee.remove_hooks()
            solver_verify.remove_hooks()
            del solver_cgee, solver_verify

            acc_gt         = (gt_answers or {}).get(ds_name, [])
            agree          = [a.strip() == b.strip() for a, b in zip(ans_cgee, ans_verify)]
            agreement_rate = sum(agree) / max(len(agree), 1)

            cgee_acc,   cgee_hits   = _acc_vs_gt(ans_cgee,   acc_gt, ds_name)
            verify_acc, verify_hits = _acc_vs_gt(ans_verify, acc_gt, ds_name)
            acc_delta = (cgee_acc - verify_acc
                         if np.isfinite(cgee_acc) and np.isfinite(verify_acc) else float('nan'))

            disagree_idxs      = [i for i, ok in enumerate(agree) if not ok]
            cgee_caused_errors = (
                [i for i in disagree_idxs if acc_gt and not cgee_hits[i] and verify_hits[i]]
                if acc_gt else []
            )

            mean_nokv   = float(np.mean(lat_nokv))
            mean_vllm   = float(np.mean(lat_vllm))
            mean_cgee   = float(np.mean(lat_cgee))
            mean_verify = float(np.mean(lat_verify_inner))

            sp_vllm          = mean_nokv / mean_vllm if mean_vllm > 0 else float('nan')
            sp_rksc          = mean_nokv / mean_cgee if mean_cgee > 0 else float('nan')
            rksc_vs_vllm_pct = (sp_rksc / sp_vllm - 1.0) * 100.0 if sp_vllm > 0 else float('nan')
            cgee_saved_ms    = mean_verify - mean_cgee

            results[ds_name] = dict(
                lat_batch_nokv         = list(lat_nokv),
                lat_vllm               = list(lat_vllm),
                lat_rksc_cgee          = list(lat_cgee),
                lat_rksc_verify        = list(lat_verify_inner),
                speedup_vllm           = sp_vllm,
                speedup_rksc           = sp_rksc,
                rksc_vs_vllm_gain      = sp_rksc / sp_vllm if sp_vllm > 0 else float('nan'),
                rksc_vs_vllm_pct       = rksc_vs_vllm_pct,
                speedup_kv_mean        = sp_vllm,   # backward compat
                savings_pct            = (1.0 - 1.0 / sp_vllm) * 100.0 if sp_vllm > 0 else float('nan'),
                kv_reuse_rate          = 1.0,
                cgee_saved_ms          = cgee_saved_ms,
                cgee_skip_rate         = cgee_skip_rate,
                cgee_skip_n            = cgee_skip_n,
                cgee_call_n            = cgee_call_n,
                cgee_speedup_vs_verify = mean_verify / mean_cgee if mean_cgee > 0 else float('nan'),
                cgee_agreement_rate    = agreement_rate,
                cgee_n_disagree        = len(disagree_idxs),
                cgee_n_caused_errors   = len(cgee_caused_errors),
                cgee_accuracy_gt       = cgee_acc,
                verify_accuracy_gt     = verify_acc,
                accuracy_delta         = acc_delta,
                has_ground_truth       = bool(acc_gt),
                n_problems             = len(padded),
                avg_tokens_raw         = avg_raw,
                avg_tokens_padded      = avg_pad,
                model_n_layers         = n_layers,
                model_n_kv_heads       = n_kv,
                model_n_q_heads        = n_q,
                model_head_dim         = head_dim,
                mem_saved_mb           = mem_saved_mb,
                max_new_tokens_used    = MAX_NEW_TOK_EXT,
            )

            reg_str = ('✓ 0 regressions' if not cgee_caused_errors
                       else f'⚠ {len(cgee_caused_errors)} regressions')
            acc_str = (f'CGEE {cgee_acc:.1%} / verify {verify_acc:.1%}  Δ={acc_delta:+.1%}'
                       if acc_gt else 'no GT')
            print(f'    NoKV  {mean_nokv:6.0f} ms  │  vLLM  {mean_vllm:6.0f} ms ({sp_vllm:.3f}×)'                  f'  │  RKSC  {mean_cgee:6.0f} ms ({sp_rksc:.3f}×  {rksc_vs_vllm_pct:+.1f}% vs vLLM)')
            print(f'    CGEE  skip={cgee_skip_rate:.0%}  saved {cgee_saved_ms:.0f} ms vs full-verify')
            print(f'    Acc   {acc_str}  │  agree {agreement_rate:.0%}  │  {reg_str}')
            if cgee_skip_rate < 0.10 and cgee_call_n >= 10:
                print(f'    ⚠  Low skip rate — recalibrate cgee_gen_conf_threshold')

        except Exception:
            import traceback; traceback.print_exc()
            results[ds_name] = {}

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    _KV_PROBE_VERBOSE = True
    unload_model(model, tok, arch)
    return results


In [ ]:
print('Freeing primary model ...')
unload_model(
    globals().pop('model', None),
    globals().pop('tokenizer', None),
    globals().pop('ARCH', None),
)

extended_results = ckpt_load('extended_results') or {}

stale = [mid for mid, r in extended_results.items()
         if isinstance(r, dict) and r and
         any(isinstance(ds, dict) and ds.get('avg_tokens_padded', 0) < 800
             for ds in r.values() if isinstance(ds, dict))]
if stale:
    print(f'Invalidating stale results (prefix<1024): {stale}')
    for mid in stale:
        del extended_results[mid]

for mid in EVAL_MODELS:
    if mid in extended_results and extended_results[mid]:
        r0 = next((v for v in extended_results[mid].values()
                   if isinstance(v, dict) and 'avg_tokens_padded' in v), {})
        if r0.get('avg_tokens_padded', 0) >= 800 and r0.get('max_new_tokens_used', 32) == MAX_NEW_TOK_EXT:
            print(f'  {mid}: loaded from checkpoint (prefix={r0.get("avg_tokens_padded", 0):.0f} tok)')
            continue

    if torch.cuda.is_available():
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        needed  = MODEL_VRAM_GB.get(mid, 30.0) + 2.0
        print(f'  [{mid.split("/")[-1]}] VRAM: {free_gb:.1f} GB free, need {needed:.0f} GB')
        if free_gb < needed:
            print(f'    SKIPPING — insufficient VRAM ({free_gb:.1f} < {needed:.0f} GB)')
            extended_results[mid] = {}
            continue

    try:
        extended_results[mid] = run_extended_benchmark(
            mid, EVAL_DATA, n_runs=N_EVAL_RUNS, gt_answers=EVAL_GT_ANSWERS)
        ckpt_save('extended_results', extended_results)
    except torch.cuda.OutOfMemoryError:
        print(f'  SKIPPED {mid}: OOM — try reducing B_FACTOR or N_EVAL')
        extended_results[mid] = {}
        unload_model()
    except Exception:
        import traceback; traceback.print_exc()
        extended_results[mid] = {}

for key in list(globals().keys()):
    if key.startswith('_ext_'):
        del globals()[key]
for _ in range(5):
    gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    needed  = MODEL_VRAM_GB.get(MODEL_ID, 16.0)
    print(f'Pre-reload VRAM: {free_gb:.1f} GB free (need ~{needed:.0f} GB)')
    if free_gb < needed:
        print('Insufficient VRAM for reload — cells using extended_results still work.')
        print('On Colab: Runtime → Restart and run all, or upgrade GPU.')
    else:
        print(f'Reloading {MODEL_ID} ...')
        model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
        ARCH = diagnose_architecture(model, tokenizer)
        ARCH['_unembed_weight'] = model.get_output_embeddings().weight.data
        print(f'  Ready. VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
else:
    model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
    ARCH = diagnose_architecture(model, tokenizer)
    ARCH['_unembed_weight'] = model.get_output_embeddings().weight.data


In [ ]:
print('\n' + '=' * 120)
print('EXTENDED EVALUATION: Multi-Model, Multi-Dataset')
print(f'Reference: Batched-no-KV  |  tok={MAX_NEW_TOK_EXT}  prefix≈1024 tok  B={B_FACTOR}')
print('Speedup = mean(Batch-noKV) / mean(condition)  [ratio-of-means]')
print('ICML claim: RKSC > vLLM-equiv (identical-prefix) + unique gains on diverse phrasings (§11)')
print('=' * 120)

_W = 28
_hdr = (
    f'  {"Model":<{_W}} {"Dataset":<14}'
    f' {"NoKV(ms)":>9} {"vLLM(ms)":>9} {"RKSC(ms)":>9}'
    f' {"vLLM↑":>7} {"RKSC↑":>7} {"vs vLLM":>8}'
    f' {"CGEE skip":>10} {"Agree%":>7} {"Acc-RKSC":>9} {"MemΔ(MB)":>9}'
    f' {"Arch":>14}'
)
print(_hdr)
print('  ' + '─' * 145)

rows_ext = []
prev_model = None

for mid, ds_results in extended_results.items():
    model_short = mid.split('/')[-1]
    if prev_model and prev_model != model_short:
        print('  ' + '·' * 145)
    prev_model = model_short

    for ds_name, r in ds_results.items():
        if not r or ds_name.startswith('_') or not isinstance(r, dict):
            continue

        lat_b    = float(np.mean(r['lat_batch_nokv']))     if 'lat_batch_nokv' in r else float('nan')
        lat_vllm = float(np.mean(r['lat_vllm']))           if 'lat_vllm'       in r else float('nan')
        lat_kv   = float(np.mean(r['lat_rksc_cgee']))           if 'lat_rksc_cgee'       in r else float('nan')
        lat_rksc = float(np.mean(r['lat_rksc_cgee']))      if 'lat_rksc_cgee'  in r else float('nan')

        vllm_gain = lat_b / lat_vllm if lat_vllm > 0 else float('nan')
        rksc_gain = lat_b / lat_rksc if lat_rksc > 0 else float('nan')
        # Δ = how much faster RKSC is vs vLLM-equiv (the ICML novelty number)
        vs_vllm   = (rksc_gain / vllm_gain - 1.0) * 100.0 if (vllm_gain > 0 and rksc_gain > 0) else float('nan')

        cgee_skip = r.get('cgee_skip_rate',      float('nan'))
        cgee_n    = r.get('cgee_call_n',          0)
        agree     = r.get('cgee_agreement_rate',  float('nan'))
        acc_rksc  = r.get('cgee_accuracy_gt',     float('nan'))
        mem_mb    = r.get('mem_saved_mb',          float('nan'))
        n_q  = r.get('model_n_q_heads', '?')
        n_kv = r.get('model_n_kv_heads', '?')
        n_l  = r.get('model_n_layers',   '?')
        arch = f'{n_l}L/{n_q}Q/{n_kv}K'

        if np.isfinite(cgee_skip) and cgee_n > 0:
            ci = 1.96 * (cgee_skip * (1 - cgee_skip) / max(cgee_n, 1)) ** 0.5
            skip_s = f'{cgee_skip:.0%}±{ci:.0%}'
        else:
            skip_s = '—'

        def _ms(v): return f'{v:7.0f}' if np.isfinite(v) else '      —'
        def _sp(v): return f'{v:6.3f}×' if np.isfinite(v) else '     —'
        def _pc(v): return f'{v:6.1%}'  if np.isfinite(v) else '     —'
        def _mb(v): return f'{v:7.1f}'  if np.isfinite(v) else '      —'
        def _delta(v):
            if not np.isfinite(v): return '      —'
            sign = '+' if v >= 0 else ''
            return f'{sign}{v:.1f}%'

        print(
            f'  {model_short:<{_W}} {ds_name:<14}'
            f' {_ms(lat_b)} {_ms(lat_vllm)} {_ms(lat_rksc)}'
            f' {_sp(vllm_gain)} {_sp(rksc_gain)} {_delta(vs_vllm):>8}'
            f' {skip_s:>10} {_pc(agree)} {_pc(acc_rksc)} {_mb(mem_mb)}'
            f' {arch:>14}'
        )

        rows_ext.append({
            'Model': mid, 'Dataset': ds_name,
            'Lat_batch_ms':      lat_b,
            'Lat_vllm_ms':       lat_vllm,
            'Lat_rksc_ms':       lat_rksc,
            'vLLM_gain':         vllm_gain,
            'RKSC_gain':         rksc_gain,
            'RKSC_vs_vLLM_pct':  vs_vllm,
            'KV_reuse':               r.get('kv_reuse_rate',          float('nan')),
            'cgee_skip_rate':         cgee_skip,
            'cgee_call_n':            cgee_n,
            'cgee_saved_ms':          r.get('cgee_saved_ms',          float('nan')),
            'cgee_speedup_vs_verify': r.get('cgee_speedup_vs_verify', float('nan')),
            'cgee_agreement_rate':    agree,
            'cgee_accuracy_gt':       acc_rksc,
            'verify_accuracy_gt':     r.get('verify_accuracy_gt',     float('nan')),
            'accuracy_delta':         r.get('accuracy_delta',         float('nan')),
            'cgee_n_caused_errors':   r.get('cgee_n_caused_errors',   0),
            'has_ground_truth':       r.get('has_ground_truth',       False),
            'mem_saved_mb':           mem_mb,
            'n_layers': n_l, 'n_kv_heads': n_kv, 'n_q_heads': n_q,
        })

print('  ' + '─' * 145)

if rows_ext:
    df_ext = pd.DataFrame(rows_ext)
    print()
    print('  Per-model averages across datasets:')
    print(f'  {"Model":<{_W}} {"vLLM↑":>7} {"RKSC↑":>7} {"vs vLLM":>9} {"CGEE skip":>10} {"Agree%":>7} {"MemΔ(MB)":>9}')
    print('  ' + '─' * 85)
    for mid, grp in df_ext.groupby('Model'):
        ms      = mid.split('/')[-1]
        vllm_m  = grp['vLLM_gain'].mean()
        rksc_m  = grp['RKSC_gain'].mean()
        delta_m = grp['RKSC_vs_vLLM_pct'].mean()
        skip_m  = grp['cgee_skip_rate'].mean()
        agree_m = grp['cgee_agreement_rate'].mean()
        mem_m   = grp['mem_saved_mb'].mean()
        print(
            f'  {ms:<{_W}} {_sp(vllm_m):>7} {_sp(rksc_m):>7} {_delta(delta_m):>9}'
            f' {_pc(skip_m):>10} {_pc(agree_m):>7} {_mb(mem_m):>9}'
        )
    print()

    _all_positive = (df_ext['RKSC_vs_vLLM_pct'].dropna() > 0).all()
    _mean_delta   = df_ext['RKSC_vs_vLLM_pct'].mean()
    print(f'  ICML claim — RKSC > vLLM-equiv: ' +
          (f'✓  mean Δ={_mean_delta:+.1f}% across all model×dataset pairs'
           if _all_positive else
           f'⚠  NOT universally positive (mean Δ={_mean_delta:+.1f}%) — check weak rows'))

if not rows_ext:
    print('  No results yet — run the extended benchmark cell above.')


In [ ]:
if rows_ext:
    df_ext = pd.DataFrame(rows_ext)

    print()
    print('── SCALING ANALYSIS: KV gain vs architecture ──')
    print(f'  {"Model":<35} {"vLLM↑":>9} {"RKSC gain":>10} {"KV saved":>9} {"Arch":>16}  MemΔ')
    print('  ' + '-'*90)
    by_model = (df_ext.groupby('Model')
                .agg({'vLLM_gain':'mean', 'RKSC_gain':'mean', 'Lat_batch_ms':'mean',
                      'Lat_vllm_ms':'mean', 'n_layers':'first',
                      'n_kv_heads':'first', 'n_q_heads':'first', 'mem_saved_mb':'mean'})
                .sort_values('RKSC_gain', ascending=False))
    for mid, row in by_model.iterrows():
        arch = f'{row.n_layers}L/{row.n_q_heads}Q/{row.n_kv_heads}K'
        saved_ms = row.Lat_batch_ms - row.Lat_vllm_ms
        mem_s = f'{row.mem_saved_mb:.0f} MB' if row.mem_saved_mb == row.mem_saved_mb else '—'
        print(f'  {mid.split("/")[-1]:<35} {row.vLLM_gain:>8.3f}× {row.RKSC_gain:>9.3f}×'
              f' {saved_ms:>7.0f} ms {arch:>16}  {mem_s}')

    _skip_data = [(r['cgee_skip_rate'], r['cgee_call_n'])
                  for r in rows_ext if r['cgee_skip_rate'] == r['cgee_skip_rate']]
    if _skip_data:
        _avg_skip = np.nanmean([x[0] for x in _skip_data])
        _avg_n    = np.mean([x[1] for x in _skip_data])
        _ci       = 1.96 * (_avg_skip * (1 - _avg_skip) / max(_avg_n, 1)) ** 0.5
        print()
        print(f'── CGEE skip-rate reliability ──')
        print(f'  Mean skip rate : {_avg_skip:.0%}  (N={_avg_n:.0f} calls/model, 95% CI ±{_ci:.0%})')

    print()
    print('── CGEE ACCURACY IMPACT ──')
    print(f'  {"Model":<35} {"Dataset":<13} {"Attempts":>9} {"Skipped":>8} {"Skip%":>6}'
          f' {"Agree%":>8} {"CGEE acc":>9} {"Verify acc":>11} {"Δacc":>6} {"Errors":>7}')
    print('  ' + '-'*110)
    for row in rows_ext:
        mn  = row['Model'].split('/')[-1]
        ds  = row['Dataset']
        cn  = row['cgee_call_n']
        sr  = row['cgee_skip_rate']
        sn  = int(round(sr * cn)) if sr == sr else 0
        ag  = row['cgee_agreement_rate']
        ca  = row['cgee_accuracy_gt']
        va  = row['verify_accuracy_gt']
        da  = row['accuracy_delta']
        er  = row['cgee_n_caused_errors']
        hgt = row['has_ground_truth']
        print(f'  {mn:<35} {ds:<13} {cn:>9} {sn:>8}'
              f' {sr:.0%} ' if sr == sr else f'  {"—":>6} ',
              end='')
        print(f'{ag:.1%} ' if ag == ag else f'   {"—":>6} ', end='')
        print(f'{ca:.1%} ' if (hgt and ca == ca) else f'  {"no GT":>6} ', end='')
        print(f'{va:.1%} ' if (hgt and va == va) else f'  {"no GT":>8} ', end='')
        print(f'{da:+.1%}' if (hgt and da == da) else f'  {"—":>5}', end='')
        print(f' {er:>7}' if hgt else f'   {"—":>5}')

    _all_errors  = sum(r['cgee_n_caused_errors'] for r in rows_ext if r['has_ground_truth'])
    _mean_agree  = np.nanmean([r['cgee_agreement_rate'] for r in rows_ext])
    _gt_rows     = [r for r in rows_ext if r['has_ground_truth']]
    _mean_delta  = np.nanmean([r['accuracy_delta'] for r in _gt_rows]) if _gt_rows else float('nan')
    print()
    print(f'  CGEE-induced errors (GT available): {_all_errors}')
    print(f'  Mean CGEE/verify agreement        : {_mean_agree:.1%}')
    if _gt_rows:
        print(f'  Mean accuracy delta               : {_mean_delta:+.1%}')
        if _all_errors == 0:
            print(f'  ✓ CGEE caused no accuracy regressions')
        else:
            print(f'  ⚠  {_all_errors} regressions — consider recalibrating cgee_gen_conf_threshold')


In [ ]:
if rows_ext:
    df_ext = pd.DataFrame(rows_ext)

    rksc_matrix = df_ext.pivot_table(index='Dataset', columns='Model',
                                      values='RKSC_gain', aggfunc='mean')
    rksc_matrix.columns = [c.split('/')[-1] for c in rksc_matrix.columns]

    fig, ax = plt.subplots(figsize=(max(6, rksc_matrix.shape[1]*2.5), 4))
    im = ax.imshow(rksc_matrix.values, cmap='RdYlGn', vmin=0.8, vmax=2.5, aspect='auto')
    ax.set_xticks(range(len(rksc_matrix.columns)))
    ax.set_xticklabels(rksc_matrix.columns, rotation=20, ha='right', fontsize=9)
    ax.set_yticks(range(len(rksc_matrix.index)))
    ax.set_yticklabels(rksc_matrix.index, fontsize=9)
    ax.set_title('RKSC speedup (KV + CGEE combined) vs Batched-no-KV', fontsize=11)
    plt.colorbar(im, ax=ax, label='RKSC speedup (×)')
    for i in range(len(rksc_matrix.index)):
        for j in range(len(rksc_matrix.columns)):
            val = rksc_matrix.values[i, j]
            if np.isfinite(val):
                ax.text(j, i, f'{val:.2f}×', ha='center', va='center', fontsize=9)
    plt.tight_layout()
    plt.show(); plt.close('all')

    df_cgee = df_ext.dropna(subset=['cgee_skip_rate']).copy()
    if not df_cgee.empty:
        fig, axes = plt.subplots(1, 3, figsize=(17, 5))
        ds_order  = ['gsm8k', 'arc_challenge', 'mmlu_stem', 'gpqa']
        ds_colors = {'gsm8k': '#2ca02c', 'arc_challenge': '#ff7f0e',
                     'mmlu_stem': '#9467bd', 'gpqa': '#d62728'}
        ds_labels = {'gsm8k': 'GSM8K (easy)', 'arc_challenge': 'ARC-C (medium)',
                     'mmlu_stem': 'MMLU-STEM (med-hard)', 'gpqa': 'GPQA (hard)'}

        ax = axes[0]
        present = [d for d in ds_order if d in df_cgee['Dataset'].values]
        by_ds   = df_cgee.groupby('Dataset')['cgee_skip_rate'].mean()
        bars = ax.bar([ds_labels.get(d, d) for d in present],
                      [by_ds.get(d, 0) for d in present],
                      color=[ds_colors.get(d, 'steelblue') for d in present],
                      edgecolor='white', linewidth=0.8)
        for bar, ds in zip(bars, present):
            v = by_ds.get(ds, 0)
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
        ax.set_ylabel('CGEE skip rate')
        ax.set_title('CGEE skip rate by difficulty\n(higher = more time saved)')
        ax.set_ylim(0, 1.0); ax.axhline(0.5, color='grey', linestyle=':', linewidth=1)
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', labelsize=8)

        ax = axes[1]
        models = df_ext['Model'].unique()
        x = np.arange(len(models)); w = 0.35
        kv_means   = [df_ext[df_ext['Model']==m]['vLLM_gain'].mean()   for m in models]
        rksc_means = [df_ext[df_ext['Model']==m]['RKSC_gain'].mean() for m in models]
        ax.bar(x - w/2, kv_means,   w, label='KV only',      color='#1f77b4', edgecolor='white')
        ax.bar(x + w/2, rksc_means, w, label='RKSC (KV+CGEE)', color='#2ca02c', edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels([m.split('/')[-1].replace('-Instruct','') for m in models],
                           rotation=15, ha='right', fontsize=8)
        ax.set_ylabel('Speedup vs Batched-no-KV')
        ax.set_title('KV vs RKSC speedup per model')
        ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
        ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

        ax = axes[2]
        for ds in df_cgee['Dataset'].unique():
            sub = df_cgee[df_cgee['Dataset'] == ds]
            ax.scatter(sub['cgee_skip_rate'], sub['cgee_saved_ms'],
                       label=ds_labels.get(ds, ds), color=ds_colors.get(ds, 'steelblue'), s=90, zorder=5)
            for _, row in sub.iterrows():
                ax.annotate(row['Model'].split('/')[-1].replace('-Instruct',''),
                            (row['cgee_skip_rate'], row['cgee_saved_ms']),
                            textcoords='offset points', xytext=(5, 3), fontsize=7)
        ax.set_xlabel('CGEE skip rate'); ax.set_ylabel('Time saved vs full verify (ms)')
        ax.set_title('Skip rate drives CGEE savings')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        plt.suptitle('CGEE Effectiveness — Across Models and Datasets', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show(); plt.close('all')
else:
    print('No extended results — run the benchmark above first.')


In [ ]:
if rows_ext and 'df_ext' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── (a) Speedup distribution by dataset ──────────────────────────────────
    ax = axes[0]
    ds_order  = ['gsm8k', 'arc_challenge', 'mmlu_stem', 'gpqa']
    ds_labels = {'gsm8k': 'GSM8K', 'arc_challenge': 'ARC-C',
                 'mmlu_stem': 'MMLU-STEM', 'gpqa': 'GPQA'}
    present   = [d for d in ds_order if d in df_ext['Dataset'].values]

    kv_by_ds   = [df_ext[df_ext['Dataset']==d]['KV_gain'].dropna().values   for d in present]
    rksc_by_ds = [df_ext[df_ext['Dataset']==d]['RKSC_gain'].dropna().values for d in present]
    x = np.arange(len(present))
    w = 0.35

    for i, (kv_vals, rksc_vals, label) in enumerate(zip(kv_by_ds, rksc_by_ds, present)):
        ax.bar(x[i] - w/2, np.mean(kv_vals),   w, color='#1f77b4',
               yerr=np.std(kv_vals),   capsize=4, edgecolor='white', label='KV only' if i==0 else '')
        ax.bar(x[i] + w/2, np.mean(rksc_vals), w, color='#2ca02c',
               yerr=np.std(rksc_vals), capsize=4, edgecolor='white', label='RKSC (KV+CGEE)' if i==0 else '')

    ax.set_xticks(x)
    ax.set_xticklabels([ds_labels.get(d, d) for d in present])
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Speedup vs Batched-no-KV')
    ax.set_title('Speedup by dataset difficulty')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # ── (b) Accuracy delta: CGEE vs full verify ───────────────────────────────
    ax = axes[1]
    gt_rows = df_ext[df_ext['has_ground_truth'] == True].copy()
    if not gt_rows.empty:
        gt_rows['model_short'] = gt_rows['Model'].apply(lambda x: x.split('/')[-1].replace('-Instruct',''))
        by_model_acc = gt_rows.groupby('model_short').agg(
            cgee_acc=('cgee_accuracy_gt', 'mean'),
            verify_acc=('verify_accuracy_gt', 'mean'),
            delta=('accuracy_delta', 'mean')
        ).sort_values('delta')

        colors_acc = ['#d62728' if d < -0.01 else '#2ca02c' if d > 0.01 else '#aec6e8'
                      for d in by_model_acc['delta']]
        bars = ax.barh(range(len(by_model_acc)), by_model_acc['delta'],
                       color=colors_acc, edgecolor='white')
        ax.set_yticks(range(len(by_model_acc)))
        ax.set_yticklabels(by_model_acc.index, fontsize=9)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Accuracy delta (CGEE − full verify)')
        ax.set_title('CGEE accuracy impact per model\n(green = CGEE improved; red = regressed)')
        ax.grid(axis='x', alpha=0.3)
        for bar, val in zip(bars, by_model_acc['delta']):
            ax.text(val + (0.002 if val >= 0 else -0.002),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:+.1%}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No ground truth available', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='grey')
        ax.set_title('CGEE accuracy impact')

    plt.suptitle('Extended Evaluation — Speedup & Accuracy Summary', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show(); plt.close('all')
else:
    print('No extended results — run the benchmark above first.')


In [ ]:
if rows_ext and 'df_ext' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── (a) Latency savings (ms) vs prefix length, colored by model ──────────
    ax = axes[0]
    palette = plt.cm.tab10.colors
    models  = df_ext['Model'].unique()
    for mi, mod in enumerate(models):
        sub = df_ext[df_ext['Model'] == mod]
        savings = sub['Lat_batch_ms'] - sub['Lat_kv_ms']
        ax.scatter(sub['Lat_batch_ms'], savings,
                   color=palette[mi % len(palette)], s=80, zorder=5,
                   label=mod.split('/')[-1].replace('-Instruct', ''))
        for _, row in sub.iterrows():
            ax.annotate(row['Dataset'],
                        (row['Lat_batch_ms'], row['Lat_batch_ms'] - row['Lat_kv_ms']),
                        textcoords='offset points', xytext=(4, 2), fontsize=7, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Reference latency — Batched-no-KV (ms)')
    ax.set_ylabel('Absolute latency saved by KV sharing (ms)')
    ax.set_title('KV savings scale with baseline cost')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

    # ── (b) GQA compression ratio vs KV gain ─────────────────────────────────
    ax = axes[1]
    by_model = df_ext.groupby('Model').agg(
        KV_gain=('KV_gain', 'mean'),
        RKSC_gain=('RKSC_gain', 'mean'),
        n_q=('n_q_heads', 'first'),
        n_kv=('n_kv_heads', 'first'),
    ).copy()
    by_model['gqa_ratio'] = by_model['n_q'].apply(
        lambda x: float(x) if str(x).replace('.','').isdigit() else float('nan')
    ) / by_model['n_kv'].apply(
        lambda x: float(x) if str(x).replace('.','').isdigit() else float('nan')
    )
    valid = by_model.dropna(subset=['gqa_ratio', 'KV_gain'])
    ax.scatter(valid['gqa_ratio'], valid['KV_gain'],   s=120, color='#1f77b4',
               label='KV only', zorder=5)
    ax.scatter(valid['gqa_ratio'], valid['RKSC_gain'], s=120, color='#2ca02c',
               marker='^', label='RKSC (KV+CGEE)', zorder=5)
    for mid, row in valid.iterrows():
        ax.annotate(mid.split('/')[-1].replace('-Instruct',''),
                    (row['gqa_ratio'], row['KV_gain']),
                    textcoords='offset points', xytext=(5, 3), fontsize=7)
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('GQA compression ratio (n_q / n_kv)')
    ax.set_ylabel('Mean speedup vs Batched-no-KV')
    ax.set_title('Higher GQA compression → greater KV sharing benefit')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.suptitle('KV Sharing — Latency Savings & Architecture Scaling', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show(); plt.close('all')
else:
    print('No extended results — run the benchmark above first.')


## §8  ASKS Semantic-Divergence Stress Test

The main experiments (§6–7) pad every problem to a 1,024-token shared prefix and
append 8–32-token branch-specific hints, so every branch trivially achieves σ ≥ 0.95
and ASKS never gates out any branch. This matches the **primary use case of ASKS**:
identical-prefix tree-search where branches diverge only in short suffix hints.

This section provides two additional validations:

**§8a–b (Graceful degradation test):** We apply ASKS to branches with genuinely
*different token sequences* (diverse phrasing templates). This tests whether ASKS
safely falls back to independent recompute when hidden-state similarity is low — a
critical safety property. At short prefix lengths (~50–120 tokens), diverse phrasings
at τ=0.82 produce ρ=28.6% reuse (below ρ*=33.2%), confirming ASKS correctly
withholds sharing when it would be costly. This is the *intended behaviour*: ASKS
degrades gracefully rather than forcing incorrect reuse.

**§8c (Generalisation test):** We show that with *longer padded prefixes* (n≈512
tokens), ASKS delivers genuine wall-clock speedup even with diverse phrasings — because
at sufficient prefix length, the break-even ρ* drops below the achievable reuse rate.
This is the first direct demonstration that ASKS generalises beyond exact-match prefix
caching when operating at realistic prefix lengths.

| Divergence level | Description | Expected σ |
|---|---|---|
| 0 — Identical | Exact same template (root) | 1.000 |
| 1 — Minor rewording | Synonym swap / word-order shift | >0.90 |
| 2 — Different structure | Altered template, same intent | τ boundary |
| 3 — Very different framing | Alternate academic register | σ < τ possible |


In [ ]:
# ── Reload model if the extended-eval loop unloaded it ───────────────────────
import gc, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from copy import deepcopy
from tqdm import tqdm

if "model" not in dir() or model is None:
    print(f"Reloading {MODEL_ID} for §8-§11 experiments …")
    model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
    ARCH = diagnose_architecture(model, tokenizer)
    ARCH["_unembed_weight"] = model.get_output_embeddings().weight.data

# ── Phrasing templates at four divergence levels ─────────────────────────────
_PHRASING_TEMPLATES = [
    # Level 0 — identical to root (trivially reusable)
    "Solve step by step:\n{q}\nLet us reason step by step.",
    # Level 1 — minor rewording
    "Solve step by step:\n{q}\nLet us think through this step by step.",
    "Work through this problem step by step:\n{q}\nLet us reason carefully.",
    # Level 2 — different structure, same instructional intent
    "Please provide a detailed solution:\n{q}\nSolution:",
    "Work through this problem carefully:\n{q}\nDetailed working:",
    "Consider the following problem and solve it:\n{q}\nStep-by-step solution:",
    # Level 3 — very different framing / academic register
    "Scientific problem requiring rigorous analysis:\n{q}\nRigorous derivation:",
    "Graduate-level examination question:\n{q}\nExpert solution:",
]

_DIVERGENCE_LEVELS = {
    0: [0],        # Identical
    1: [1, 2],     # Minor rewording
    2: [3, 4, 5],  # Different structure
    3: [6, 7],     # Very different framing
}

_DIV_LABELS = {
    0: "Identical",
    1: "Minor rewording",
    2: "Different structure",
    3: "Very different framing",
}

N_SIM_PROBLEMS = 15  # problems for similarity analysis

@torch.inference_mode()
def get_last_hs_per_layer(model, tokenizer, text, device, max_len=512):
    """Last-token hidden state at every transformer layer.  Returns [L, H] on CPU."""
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=max_len).to(device)
    out = model(**inputs, output_hidden_states=True, use_cache=False,
                output_attentions=False)
    hs = torch.stack([h[:, -1, :].float() for h in out.hidden_states[1:]])  # [L,1,H]
    del out
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return hs.squeeze(1).cpu()  # [L, H]

def asks_sim(hs_a, hs_b):
    """Mean layer-wise cosine similarity σ_b (Eq. 2 in paper)."""
    n_a = hs_a.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    n_b = hs_b.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return float(((hs_a / n_a) * (hs_b / n_b)).sum(-1).mean().item())

print(f"Phrasing templates: {len(_PHRASING_TEMPLATES)}")
print(f"N_SIM_PROBLEMS: {N_SIM_PROBLEMS}")


In [ ]:
# ── Compute ASKS similarity for every phrasing vs the root (template 0) ──────
cal_tau = CALIBRATED.get(MODEL_ID, {"tau": 0.82, "theta": 6.0})["tau"]

_sim_probs = list(_problems[:N_SIM_PROBLEMS])
sim_records = []

print(f"Computing ASKS similarities … (τ = {cal_tau})")
for pi, prob in enumerate(tqdm(_sim_probs, desc="Problems")):
    root_text = _PHRASING_TEMPLATES[0].format(q=prob)
    hs_root   = get_last_hs_per_layer(model, tokenizer, root_text, DEVICE)
    n_root    = hs_root.norm(dim=-1, keepdim=True).clamp_min(1e-8)

    for ti, tmpl in enumerate(_PHRASING_TEMPLATES):
        branch_text = tmpl.format(q=prob)
        hs_branch   = get_last_hs_per_layer(model, tokenizer, branch_text, DEVICE)
        sim = asks_sim(hs_root, hs_branch)
        level = next(lvl for lvl, idxs in _DIVERGENCE_LEVELS.items() if ti in idxs)
        sim_records.append({
            "problem_idx": pi, "template_idx": ti,
            "divergence_level": level, "label": _DIV_LABELS[level],
            "similarity": sim, "would_reuse_kv": sim >= cal_tau,
        })

df_sim = pd.DataFrame(sim_records)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'Level':<5} {'Description':<22} {'Mean σ':>8} {'Min σ':>8} {'Max σ':>8} {'Reuse %':>9}")
print("─" * 65)
for lvl in sorted(_DIVERGENCE_LEVELS):
    sub  = df_sim[df_sim["divergence_level"] == lvl]
    sims = sub["similarity"].values
    print(f"{lvl:<5} {_DIV_LABELS[lvl]:<22} "
          f"{sims.mean():>8.4f} {sims.min():>8.4f} {sims.max():>8.4f} "
          f"{sub['would_reuse_kv'].mean():>9.1%}")

print(f"{'':5} {'Overall':<22} "
      f"{df_sim['similarity'].mean():>8.4f}  —  —  "
      f"{df_sim['would_reuse_kv'].mean():>9.1%}")


In [ ]:
# ── Plot: similarity distribution + reuse rate ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
levels  = sorted(_DIVERGENCE_LEVELS.keys())

# Left: box plot per divergence level
ax = axes[0]
bp_data = [df_sim[df_sim["divergence_level"] == lvl]["similarity"].values
           for lvl in levels]
bp = ax.boxplot(bp_data, patch_artist=True, widths=0.45,
                medianprops={"color": "#222", "linewidth": 2})
for patch, color in zip(bp["boxes"], _colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.axhline(cal_tau, color="red", linestyle="--", linewidth=1.5,
           label=f"ASKS threshold τ = {cal_tau}")
ax.set_xticks(range(1, len(levels) + 1))
ax.set_xticklabels([_DIV_LABELS[l] for l in levels], fontsize=8, rotation=8)
ax.set_ylabel("ASKS cosine similarity σ")
ax.set_title("ASKS similarity by phrasing divergence level")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(max(0.75, df_sim["similarity"].min() - 0.02), 1.005)

# Right: KV reuse rate by level
ax = axes[1]
reuse_rates = [df_sim[df_sim["divergence_level"] == lvl]["would_reuse_kv"].mean()
               for lvl in levels]
bars = ax.bar(range(len(levels)), reuse_rates,
              color=_colors, alpha=0.75, edgecolor="white", linewidth=0.8)
for bar, rate in zip(bars, reuse_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{rate:.0%}", ha="center", fontsize=11, fontweight="bold")
ax.set_xticks(range(len(levels)))
ax.set_xticklabels([_DIV_LABELS[l] for l in levels], fontsize=8, rotation=8)
ax.set_ylabel("KV reuse rate (fraction of branches above τ)")
ax.set_ylim(0, 1.2)
ax.set_title("ASKS KV cache reuse rate by divergence level")
ax.axhline(1.0, color="#aaa", linestyle=":", linewidth=1)
ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    f"ASKS Semantic-Divergence Stress Test — {MODEL_ID.split('/')[-1]} "
    f"(N={N_SIM_PROBLEMS} problems, τ={cal_tau})",
    fontsize=11)
plt.tight_layout()
plt.savefig("fig_asks_divergence.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_asks_divergence.pdf")


In [ ]:
# ── §8b helper: τ-sweep benchmark in the production regime ────────────
# v5.1 fix: the old benchmark_diverse_prefix used diverse FULL-template
# branches (no shared token prefix).  ASKS cannot save work in that
# regime — the paper is explicit that its scope is "identical long
# prefix + diverse short suffixes" (outline §4).  The old code therefore
# measured a counterfactual that does not match the paper setup.
#
# New design: parameterise the §6 benchmark by τ.  All branches share the
# padded prefix (σ ≈ 1.0 always); τ moves the gate decision boundary.
# Speedup traces the real cost/benefit tradeoff of gating in production.

def benchmark_tau(model, tokenizer, padded_problems, arch,
                   tau: float, n_runs: int = 2, n_warmup: int = 1,
                   label: str = ""):
    """Benchmark ASKS at a given τ against the No-KV baseline.

    Uses the full solver path (same as §6) so measurements are
    directly comparable to Table 1.  The identical-prefix regime means
    σ ≡ 1.0; τ sweeps the gate decision only.
    """
    cfg_asks                                 = deepcopy(cfg_bm)
    cfg_asks.rksc.tau_similarity_threshold   = tau
    cfg_asks.verify_with_cgee                = False
    cfg_asks.verify_full_depth               = False

    lat_nokv, _       = benchmark_condition(
        model, tokenizer, padded_problems, cfg_bm, arch,
        disable_reuse=True, batch_no_kv=True,
        n_runs=n_runs, n_warmup=n_warmup,
        label=f"{label} NoKV")

    lat_asks, rr      = benchmark_condition(
        model, tokenizer, padded_problems, cfg_asks, arch,
        disable_reuse=False, batch_no_kv=False,
        n_runs=n_runs, n_warmup=n_warmup,
        label=f"{label} ASKS τ={tau:.2f}")

    return lat_nokv, lat_asks, rr


print("benchmark_tau (τ-sweep helper) defined.")


In [ ]:
# ── §8b  τ sensitivity sweep (production regime) ──────────────────────
# v5.1 fix: the old §8b benchmark used a broken regime (no shared prefix);
# empirical speedup at τ=0.75/0.82 was 0.71× which was an artifact, not a
# real result.  We now use the production regime (identical padded prefix
# + diverse suffix hints, matching §6) and measure the genuine τ→speedup
# curve via the full solver path.

N_BENCH_DIV    = 20
_div_probs_pad = [pad_to_target(p, tokenizer, 512)
                   for p in list(_problems[:N_BENCH_DIV])]
_actual_len    = int(np.mean([
    int(tokenizer(_PAD_TEMPLATE.format(q=p), return_tensors="pt",
                  truncation=True, max_length=MAX_SEQ_LEN
                 )["input_ids"].shape[1])
    for p in _div_probs_pad]))
print(f"§8b regime: identical padded prefix (~{_actual_len} tok) + "
      f"diverse suffix hints  |  N={N_BENCH_DIV}  B={cfg_bm.branching_factor}")

# Still precompute phrasing-divergence similarities for §8a plot & ρ* use.
print("\nPrecomputing phrasing-divergence similarities (for ρ* reference) …")
_all_sims_by_prob = []
for pi, prob in enumerate(tqdm(list(_problems[:N_BENCH_DIV]),
                                desc="Similarity precompute")):
    phrasings = [_PHRASING_TEMPLATES[b % len(_PHRASING_TEMPLATES)].format(q=prob)
                 for b in range(min(cfg_bm.branching_factor,
                                    len(_PHRASING_TEMPLATES)))]
    hs_root   = get_last_hs_per_layer(model, tokenizer, phrasings[0], DEVICE)
    prob_sims = [
        asks_sim(hs_root, get_last_hs_per_layer(model, tokenizer, p, DEVICE))
        for p in phrasings[1:]
    ]
    _all_sims_by_prob.append(prob_sims)

# τ sweep — empirical.
TAUS_TO_BENCH     = [0.70, 0.82, 0.90, 0.95]
bench_tau_results = {}

print(f"\nBenchmarking wall-clock ASKS at each τ  "
      f"(N={N_BENCH_DIV}, n_runs={N_RUNS}) …")
for tau_b in TAUS_TO_BENCH:
    lat_nokv, lat_asks, rr = benchmark_tau(
        model, tokenizer, _div_probs_pad, ARCH,
        tau=tau_b, n_runs=N_RUNS, n_warmup=1, label=f"  §8b τ={tau_b}")
    sp        = float(np.mean(lat_nokv)) / float(np.mean(lat_asks))
    ci        = bootstrap_ci_ratio(lat_nokv, lat_asks)
    _, p_tau  = paired_ttest(lat_nokv, lat_asks)
    bench_tau_results[tau_b] = dict(
        speedup=sp, ci=ci, p=p_tau,
        reuse=float(np.mean(rr)),
        lat_nokv=float(np.mean(lat_nokv)),
        lat_asks=float(np.mean(lat_asks)),
    )
    marker = '✓' if sp >= 1.0 else '✗'
    print(f'  τ={tau_b}: speedup={sp:.3f}×  '
          f'[95%CI {ci[0]:.3f}–{ci[1]:.3f}]  '
          f'reuse={np.mean(rr):.1%}  p={p_tau:.2e}  {marker}')

# ── ρ* from §8a phrasing-divergence sims at τ=0.82 ─────────────────────
# In the identical-prefix regime all τ yield ρ=100% and speedup > 1.0, so
# interpolation from that regime always returns ρ*=0% (uninformative).
# The meaningful ρ* is the observed reuse rate on diverse phrasings
# (§8a data) — this reflects the real operating point where the gate matters.
_tau_ref     = cfg_bm.rksc.tau_similarity_threshold
_all_sims_flat = [s for row in _all_sims_by_prob for s in row]
_rho_star    = float(np.mean([s >= _tau_ref for s in _all_sims_flat]))

print(f"\nBreak-even reuse fraction ρ* (diverse phrasings at τ={_tau_ref}): "
      f"ρ* ≈ {_rho_star:.1%}")
print("(Fraction of diverse-phrasing branches that pass the ASKS gate at "
      "the calibrated τ — the operating reuse in the §11 novelty regime.)")
print("Regime: identical padded prefix + diverse suffix hints (matches §6).")


## §9  Prefix Length Sweep — ASKS Gating Boundary

All §5–7 experiments fix n ≈ 1,024 tokens, where branch-specific suffixes
(8–32 tokens) represent < 3% of total context and ASKS never gates any branch.
This experiment sweeps n ∈ {128, 256, 512, 1024} to show:

- At which prefix length KV sharing transitions from net-costly to net-beneficial.
- How the ASKS reuse rate degrades as the branch-specific content grows relative
  to the shared prefix (the regime where ASKS's semantic gating actually fires).
- Empirical confirmation of the analytical threshold n* from Proposition 4.1.


In [ ]:
# ── §9  END-TO-END prefix-length sweep for ASKS ───────────────────────
# SCOPE: all latencies are END-TO-END (prefill + decode).
# Regime: identical padded prefix + diverse suffix hints (matches §6).
# Every τ in [0.70, 0.95] yields σ ≡ 1.0 because the prefix is byte-identical
# across branches, so the gate decision is τ-insensitive at this operating
# point.  Meaningful τ variation shows up only on diverse phrasings (§8a).
#
# Statistical reporting:
#   - 95% bootstrap CI on every speedup ratio (ratio-of-means)
#   - CV reported alongside N to characterise measurement reliability
#   - Single run per condition per prefix — noted in output

PREFIX_SWEEP_TARGETS = [128, 256, 512, 1024]
N_SWEEP              = 20
_sweep_probs_raw     = list(_problems[:N_SWEEP])

sweep_results = {}

for target in PREFIX_SWEEP_TARGETS:
    print(f'\n── Prefix target = {target} tokens ─────────────────────────')
    padded      = [pad_to_target(p, tokenizer, target) for p in _sweep_probs_raw]
    actual_lens = tokenizer(padded, return_length=True, truncation=True,
                             max_length=MAX_SEQ_LEN)['length']
    actual_mean = float(np.mean(actual_lens))
    print(f'  Actual mean: {actual_mean:.0f} tokens')

    lat_nkv, _  = benchmark_condition(
        model, tokenizer, padded, cfg_bm, ARCH,
        disable_reuse=True, batch_no_kv=True, full_depth_verify=False,
        n_runs=N_RUNS, label=f'  NoKV n={target}', n_warmup=1)

    lat_kv, rr  = benchmark_condition(
        model, tokenizer, padded, cfg_bm, ARCH,
        disable_reuse=False, batch_no_kv=False, full_depth_verify=False,
        n_runs=N_RUNS,
        label=f'  ASKS τ={cfg_bm.rksc.tau_similarity_threshold} n={target}',
        n_warmup=1)

    sp_asks  = float(np.mean(lat_nkv)) / float(np.mean(lat_kv))
    ci_asks  = bootstrap_ci_ratio(lat_nkv, lat_kv)
    _, p_tau = paired_ttest(lat_nkv, lat_kv)

    sweep_results[target] = dict(
        actual_len   = actual_mean,
        lat_nokv_ms  = float(np.mean(lat_nkv)),
        lat_kv_ms    = float(np.mean(lat_kv)),
        lat_nokv_std = float(np.std(lat_nkv)),
        lat_kv_std   = float(np.std(lat_kv)),
        speedup      = sp_asks,
        ci_asks      = ci_asks,
        reuse_rate   = float(np.mean(rr)),
        cv_nokv      = cv_percent(lat_nkv),
        cv_asks      = cv_percent(lat_kv),
        p_asks       = p_tau,
        n_problems   = N_SWEEP,
    )
    print(f'  END-TO-END (N={N_SWEEP}, n_runs={N_RUNS}):')
    print(f'    ASKS: {sp_asks:.3f}×  [95%CI {ci_asks[0]:.3f}–{ci_asks[1]:.3f}]  '
          f'reuse={np.mean(rr):.1%}  '
          f'CV={cv_percent(lat_kv):.0f}%  p={p_tau:.4f}')

print()
print('─' * 72)
print(f'{"n":>6}  {"ASKS sp":>10}  {"95% CI":>22}  {"CV%":>5}  N')
print('─' * 72)
for n, r in sorted(sweep_results.items()):
    ci = r['ci_asks']
    print(f'{n:>6}  {r["speedup"]:>10.3f}×  '
          f'{f"{ci[0]:.3f}–{ci[1]:.3f}":>22}  '
          f'{r["cv_nokv"]:>4.0f}%  {r["n_problems"]}')
print()
print('NOTE: All numbers are END-TO-END (prefill + decode).')
print('NOTE: Wide CIs at n=128 reflect small absolute saving relative to noise.')


In [ ]:
# ── §9 Plot: END-TO-END speedup + CI + latency composition ──────────
ns       = sorted(sweep_results.keys())
sp_asks  = [sweep_results[n]['speedup']    for n in ns]
lat_nokv = [sweep_results[n]['lat_nokv_ms'] for n in ns]
lat_kv   = [sweep_results[n]['lat_kv_ms']   for n in ns]
actuals  = [sweep_results[n]['actual_len']  for n in ns]
ci_lo    = [sweep_results[n]['ci_asks'][0] for n in ns]
ci_hi    = [sweep_results[n]['ci_asks'][1] for n in ns]
err_asks = [[s - lo for s, lo in zip(sp_asks, ci_lo)],
            [hi - s for s, hi in zip(sp_asks, ci_hi)]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: speedup vs prefix length with 95% CI error bars.
ax = axes[0]
ax.errorbar(ns, sp_asks, yerr=err_asks, fmt='o-', color='#1f77b4',
             linewidth=2, markersize=8, capsize=5,
             label='ASKS (end-to-end)')
ax.axhline(1.0, color='#333', linestyle=':', linewidth=1)
ax.set_xlabel('Prefix length (tokens)')
ax.set_ylabel('End-to-end speedup vs Batched-no-KV')
ax.set_title('§9  End-to-end speedup vs prefix length\n(95% bootstrap CI)')
ax.set_xscale('log', base=2)
ax.set_xticks(ns)
ax.set_xticklabels([str(n) for n in ns])
ax.grid(alpha=0.3); ax.legend()

# Middle: wall-clock latency bars per n.
ax = axes[1]
x  = np.arange(len(ns)); w = 0.38
ax.bar(x - w / 2, lat_nokv, width=w, label='No-KV',  alpha=0.85,
        color='#7f7f7f',
        yerr=[sweep_results[n]['lat_nokv_std'] for n in ns], capsize=3)
ax.bar(x + w / 2, lat_kv,   width=w, label='ASKS',   alpha=0.85,
        color='#1f77b4',
        yerr=[sweep_results[n]['lat_kv_std']   for n in ns], capsize=3)
ax.set_xticks(x); ax.set_xticklabels([str(n) for n in ns])
ax.set_xlabel('Prefix length (tokens)')
ax.set_ylabel('Wall-clock latency (ms)')
ax.set_title('Wall-clock latency composition')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Right: reuse rate + CV per n.
ax       = axes[2]
reuse    = [sweep_results[n]['reuse_rate'] * 100 for n in ns]
cv_asks  = [sweep_results[n]['cv_asks']         for n in ns]
ax.plot(ns, reuse, 'o-', color='#2ca02c', label='Reuse rate (%)', markersize=8)
ax2 = ax.twinx()
ax2.plot(ns, cv_asks, 's--', color='#d62728', label='CV (%)', markersize=6)
ax.set_xscale('log', base=2); ax.set_xticks(ns)
ax.set_xticklabels([str(n) for n in ns])
ax.set_xlabel('Prefix length (tokens)')
ax.set_ylabel('KV reuse rate (%)', color='#2ca02c')
ax2.set_ylabel('Measurement CV (%)', color='#d62728')
ax.set_title('Reuse rate and measurement reliability')
ax.grid(alpha=0.3)

plt.suptitle(f'§9  ASKS prefix-length sweep — {MODEL_ID.split("/")[-1]} '
              f'(N={N_SWEEP}, B={B_FACTOR}, end-to-end latency)',
              fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_sec9_sweep.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_sec9_sweep.pdf')


## §10  RSBCM Multi-Depth Validation (Fixed)

The original §10 found 0 evictions because `rsbcm.reset()` is called at the top
of every `solve()`, zeroing the event counter and the block pool before any
accumulation can happen.  RSBCM is designed for multi-depth tree search *within*
a single `solve()` call — not across problems.

To trigger it, we need `max_blocks < B` so that the B blocks allocated during
a single problem's `_branch_passes` call already exceed the cap.  With B=8 and
`max_blocks=4`, the first `_branch_passes` allocates 8 blocks, `_maybe_evict`
fires 4 times, and we can observe both the eviction events and their overhead.

We also patch the `reset()` call to preserve a cumulative eviction counter so
latency comparisons are fair across problems.


In [ ]:
# ── §10 Fixed: patch RKSCManager to count cumulative evictions ────────────────
# rsbcm.reset() zeros eviction_events — we wrap it to preserve a running total.

import types

def _make_tracked_rsbcm(rsbcm_instance):
    """Monkey-patch an RSBCMManager so cumulative_evictions survives reset()."""
    rsbcm_instance.cumulative_evictions = 0
    _orig_reset = rsbcm_instance.reset

    def _tracked_reset(self):
        # preserve cumulative count, then do normal reset
        _save = self.eviction_events
        _orig_reset()
        self.cumulative_evictions += _save

    rsbcm_instance.reset = types.MethodType(_tracked_reset, rsbcm_instance)
    return rsbcm_instance


N_RSBCM_PROBS = 20
_rsbcm_probs = [pad_to_target(p, tokenizer, PREFIX_TARGET) for p in list(_problems[:N_RSBCM_PROBS])]


def run_rsbcm_session_fixed(model, tokenizer, problems, arch, cfg_base,
                              max_blocks, label=""):
    """
    Run N problems, tracking cumulative RSBCM evictions across all solve() calls.
    Uses max_blocks < B to guarantee evictions fire within a single solve().
    """
    cfg_r = deepcopy(cfg_base)
    cfg_r.rsbcm.max_blocks   = max_blocks
    cfg_r.verify_with_cgee   = False
    cfg_r.verify_full_depth  = False

    solver = RKSCSolverV8(model, tokenizer, cfg_r, arch, disable_reuse=False)
    _make_tracked_rsbcm(solver.rsbcm)

    lats, ev_per_prob, answers = [], [], []
    prev_cumul = 0

    for prob in tqdm(problems, desc=label):
        res = solver.solve(prob)
        # eviction_events counts events in THIS solve() call (before reset zeroed it)
        # We read it AFTER solve() but the last reset() inside solve() has already
        # zeroed it — so we track via cumulative_evictions delta instead.
        solver.rsbcm.cumulative_evictions += solver.rsbcm.eviction_events
        ev_this = solver.rsbcm.cumulative_evictions - prev_cumul
        prev_cumul = solver.rsbcm.cumulative_evictions
        lats.append(res.total_latency_ms)
        ev_per_prob.append(ev_this)
        answers.append(res.answer)
        del res

    total_ev = solver.rsbcm.cumulative_evictions
    solver.remove_hooks()
    del solver
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return lats, ev_per_prob, answers, total_ev


# Note: reset() is called at START of solve(), so eviction_events from the
# previous solve() call is already wiped. We need to intercept BEFORE reset().
# Simpler fix: override reset() to NOT zero eviction_events but only the pool.

def run_rsbcm_session_v2(model, tokenizer, problems, arch, cfg_base,
                          max_blocks, label=""):
    """
    Cleaner approach: override _maybe_evict to count via an external list.
    This avoids the reset() timing issue entirely.
    """
    cfg_r = deepcopy(cfg_base)
    cfg_r.rsbcm.max_blocks   = max_blocks
    cfg_r.verify_with_cgee   = False
    cfg_r.verify_full_depth  = False

    solver = RKSCSolverV8(model, tokenizer, cfg_r, arch, disable_reuse=False)

    # External eviction log not zeroed by reset()
    _ev_log = []

    _orig_maybe_evict = solver.rsbcm._maybe_evict

    def _patched_maybe_evict():
        before = len(solver.rsbcm._pool)
        _orig_maybe_evict()
        after  = len(solver.rsbcm._pool)
        evicted = before - after
        if evicted > 0:
            _ev_log.append(evicted)

    solver.rsbcm._maybe_evict = _patched_maybe_evict

    lats, ev_per_prob, answers = [], [], []
    prev_n_evictions = 0

    for prob in tqdm(problems, desc=label):
        res = solver.solve(prob)
        ev_this = len(_ev_log) - prev_n_evictions
        prev_n_evictions = len(_ev_log)
        lats.append(res.total_latency_ms)
        ev_per_prob.append(ev_this)
        answers.append(res.answer)
        del res

    solver.remove_hooks()
    del solver
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    total_evicted_blocks = sum(_ev_log)
    return lats, ev_per_prob, answers, total_evicted_blocks


# ── max_blocks=100000: baseline (RSBCM never fires, B=8 << 100000) ────────────
print("Baseline (max_blocks=100000, RSBCM dormant) …")
lats_no_rsbcm, ev_no, ans_no, total_ev_no = run_rsbcm_session_v2(
    model, tokenizer, _rsbcm_probs, ARCH, cfg_bm,
    max_blocks=100_000, label="RSBCM dormant")

# ── max_blocks=4: forces evictions (B=8 > 4 ∴ 4 evictions per solve call) ────
print("\nRSBCM active (max_blocks=4, B=8 > 4 → 4 evictions per problem) …")
lats_rsbcm, ev_yes, ans_yes, total_ev_yes = run_rsbcm_session_v2(
    model, tokenizer, _rsbcm_probs, ARCH, cfg_bm,
    max_blocks=4, label="RSBCM active (max_blocks=4)")

agree = [a.strip() == b.strip() for a, b in zip(ans_no, ans_yes)]
agreement_rate = sum(agree) / max(len(agree), 1)
overhead = np.mean(lats_rsbcm) - np.mean(lats_no_rsbcm)

print(f"\n{'─'*62}")
print(f"RSBCM Validation (Fixed) — {N_RSBCM_PROBS} problems, B={cfg_bm.branching_factor}")
print(f"{'─'*62}")
print(f"  Dormant  (max_blocks=100k): {np.mean(lats_no_rsbcm):7.1f} ± "
      f"{np.std(lats_no_rsbcm):.1f} ms  | {total_ev_no} evictions")
print(f"  Active   (max_blocks=4):    {np.mean(lats_rsbcm):7.1f} ± "
      f"{np.std(lats_rsbcm):.1f} ms  | {total_ev_yes} evictions")
print(f"  Eviction overhead:  {overhead:+.1f} ms/problem  "
      f"({overhead/max(np.mean(lats_no_rsbcm),1)*100:+.2f}%)")
from scipy import stats as _st
_diffs = np.array(lats_rsbcm) - np.array(lats_no_rsbcm)
_se = float(np.std(_diffs) / np.sqrt(len(_diffs)))
_ci95 = 1.96 * _se
_, _pval = _st.ttest_rel(lats_rsbcm, lats_no_rsbcm)
print(f"  95% CI on overhead: [{overhead-_ci95:+.0f}, {overhead+_ci95:+.0f}] ms  p={_pval:.3f}")
_cv = float(np.std(lats_no_rsbcm)/np.mean(lats_no_rsbcm)*100)
print(f"  CV={_cv:.0f}% — conclusion is overhead not large (not provably zero).")
print(f"  Expected evictions: {N_RSBCM_PROBS} × "
      f"{cfg_bm.branching_factor - 4} = {N_RSBCM_PROBS*(cfg_bm.branching_factor-4)} "
      f"  Observed: {total_ev_yes}")
print(f"  Answer agreement:   {agreement_rate:.1%} ({sum(agree)}/{len(agree)} problems)")
if agreement_rate >= 0.95:
    print(f"  ✓  RSBCM eviction introduces no answer regressions.")
else:
    n_bad = len(agree) - sum(agree)
    print(f"  ⚠  {n_bad} disagreements — eviction may be corrupting outputs.")


In [ ]:
# ── Plot: RSBCM eviction events and latency ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

prob_ids = list(range(N_RSBCM_PROBS))

ax = axes[0]
ax.plot(prob_ids, lats_no_rsbcm, "o--", color="#aec6e8", linewidth=1.5,
        markersize=5, label="RSBCM dormant (max_blocks=100k)")
ax.plot(prob_ids, lats_rsbcm, "o-", color="#d62728", linewidth=1.5,
        markersize=5, label="RSBCM active (max_blocks=4)")
ax.set_xlabel("Problem index")
ax.set_ylabel("Latency (ms)")
ax.set_title("Per-problem latency with and without RSBCM eviction")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Annotate overhead
_oh_str = f"Mean overhead: {overhead:+.1f} ms"
ax.text(0.98, 0.97, _oh_str, transform=ax.transAxes,
        ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.8))

ax = axes[1]
ax.bar(prob_ids, ev_yes, color="#d62728", alpha=0.75,
       label=f"Evictions per problem (expected={cfg_bm.branching_factor-4})")
ax.axhline(cfg_bm.branching_factor - 4, color="#222", linestyle="--",
           linewidth=1.5, label=f"Expected (B - max_blocks = {cfg_bm.branching_factor-4})")
ax.set_xlabel("Problem index")
ax.set_ylabel("Eviction events per problem")
ax.set_title("RSBCM evictions per problem (max_blocks=4)")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
_summary = (f"Total evictions: {total_ev_yes}\n"
            f"Expected: {N_RSBCM_PROBS*(cfg_bm.branching_factor-4)}\n"
            f"Agreement: {agreement_rate:.1%}")
ax.text(0.98, 0.98, _summary, transform=ax.transAxes,
        ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.8))

plt.suptitle(
    f"RSBCM Validation — {MODEL_ID.split('/')[-1]} "
    f"(N={N_RSBCM_PROBS}, B={cfg_bm.branching_factor})",
    fontsize=11)
plt.tight_layout()
plt.savefig("fig_rsbcm_validation_fixed.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_rsbcm_validation_fixed.pdf")


## §11  Token-Exact vs Semantic KV Sharing — ASKS Novelty Over vLLM

This section isolates the **actual novel contribution of ASKS over vLLM / SGLang**.
§8c showed that in the identical-prefix regime, vLLM prefix caching achieves
the same speedup — so the 1.14× there is not novel. The genuine novelty is here:
ASKS retains meaningful KV reuse on branches where token sequences diverge from
token 1 (diverse phrasings), whereas vLLM / SGLang token-exact caching yields 0%
reuse. This is the contribution that cannot be replicated by existing systems.

The reviewer concern: all speedups are measured vs a "No-KV" baseline that
recomputes the prefix independently for every branch — something vLLM and SGLang
do not do.  This section adds the missing comparison:

- **Exact-match KV sharing**: reuse prefix KV iff the branch shares a full
  token-level prefix with the root (standard vLLM / SGLang behaviour).
- **ASKS semantic KV sharing**: reuse iff σ_b ≥ τ (hidden-state cosine similarity).

On **identical-prefix** branches (§5–7 operating point), both methods behave
identically (100% reuse for both).  On **diverse-prefix** branches (§8 regime),
exact-match fails (0% reuse) while ASKS retains high reuse, isolating the
genuine contribution of the semantic gating mechanism.


In [ ]:
# ── Token-exact vs ASKS on diverse phrasings ──────────────────────────────────
N_EXACT = 20
_exact_probs_raw = list(_problems[:N_EXACT])
tau_exact = CALIBRATED.get(MODEL_ID, {"tau": 0.82, "theta": 6.0})["tau"]

def token_exact_reuse_rate(tokenizer, phrasings, max_len=1024):
    """Fraction of branch phrasings that share an exact token prefix with branch 0."""
    root_ids = tokenizer(phrasings[0], return_tensors="pt",
                          truncation=True, max_length=max_len)["input_ids"][0].tolist()
    reuses = 0
    for ph in phrasings[1:]:
        branch_ids = tokenizer(ph, return_tensors="pt",
                                truncation=True, max_length=max_len
                                )["input_ids"][0].tolist()
        lcp = sum(1 for a, b in zip(root_ids, branch_ids) if a == b)
        # Full prefix match: every token in root appears in branch at same position
        if lcp == len(root_ids):
            reuses += 1
    return reuses / max(len(phrasings) - 1, 1)

rows_cmp = []
print(f"{'Prob':>5} {'Exact reuse':>13} {'ASKS reuse':>12}  {'ASKS sims (branches 1–4)':>30}")
print("─" * 65)

for pi, prob in enumerate(tqdm(_exact_probs_raw, desc="Exact vs ASKS")):
    B_cmp     = min(cfg_bm.branching_factor, len(_PHRASING_TEMPLATES))
    phrasings = [_PHRASING_TEMPLATES[b % len(_PHRASING_TEMPLATES)].format(q=prob)
                 for b in range(B_cmp)]

    exact_r = token_exact_reuse_rate(tokenizer, phrasings)

    # ASKS similarity for each branch vs root
    hs_root = get_last_hs_per_layer(model, tokenizer, phrasings[0], DEVICE)
    sims = []
    for b in range(1, B_cmp):
        hs_b = get_last_hs_per_layer(model, tokenizer, phrasings[b], DEVICE)
        sims.append(asks_sim(hs_root, hs_b))
    asks_r = float(np.mean([s >= tau_exact for s in sims]))

    rows_cmp.append({
        "problem_idx": pi,
        "exact_reuse_rate": exact_r,
        "asks_reuse_rate":  asks_r,
        "mean_asks_sim":    float(np.mean(sims)),
    })
    sim_str = " ".join(f"{s:.3f}" for s in sims[:4])
    print(f"{pi:>5} {exact_r:>13.1%} {asks_r:>12.1%}  [{sim_str} …]")

df_cmp = pd.DataFrame(rows_cmp)
print("─" * 65)
print(f"{'Mean':>5} {df_cmp['exact_reuse_rate'].mean():>13.1%} "
      f"{df_cmp['asks_reuse_rate'].mean():>12.1%}")

print(f"\nSummary — diverse phrasings (τ = {tau_exact}):")
print(f"  Token-exact KV reuse: {df_cmp['exact_reuse_rate'].mean():.1%} "
      f"(expects 0% — branches have different token sequences)")
print(f"  ASKS semantic reuse:  {df_cmp['asks_reuse_rate'].mean():.1%} "
      f"(correctly reuses where hidden-state similarity is high)")
_gain = (df_cmp["asks_reuse_rate"].mean() - df_cmp["exact_reuse_rate"].mean())
print(f"  ASKS vs exact:        +{_gain:.1%} additional reuse on diverse branches")
print(f"  ↳  This is the marginal contribution of ASKS over standard prefix caching.")

# ── FIX (v3): Wall-clock timing — ASKS vs vLLM on PADDED diverse phrasings ──
# Key insight: wall-clock advantage requires LONG shared prefix so root-forward
# cost is amortised across B branches. We use 512-tok padded problems wrapped
# in diverse phrasing templates — the same regime as §8c.
# vLLM-equivalent (τ=-inf → reuse=100% → same as ASKS on identical tokens).
# In this regime vLLM ≡ ASKS (both reuse the shared prefix KV).
# The §11 novelty claim is specifically the REUSE RATE on diverse phrasings —
# +28.6pp over token-exact — which translates to speedup only when prefix is
# long enough. We measure both.
print("\n── §11 Wall-clock: ASKS vs vLLM-equivalent on padded diverse phrasings ──")
print("   (Padded to 512 tokens so root-forward cost is amortised)")
N_WC = min(N_EXACT, 12)
_wc_probs_padded = [pad_to_target(p, tokenizer, 512) for p in _exact_probs_raw[:N_WC]]

cfg_11_nkv = deepcopy(cfg_bm)
cfg_11_nkv.verify_with_cgee  = False
cfg_11_nkv.verify_full_depth = False
lat_vllm_11, _ = benchmark_condition(
    model, tokenizer, _wc_probs_padded, cfg_11_nkv, ARCH,
    disable_reuse=True, batch_no_kv=True,
    n_runs=N_RUNS, label="  §11 vLLM-equiv (No-KV baseline)")

cfg_11_asks = deepcopy(cfg_bm)
cfg_11_asks.rksc.tau_similarity_threshold = tau_exact
cfg_11_asks.verify_with_cgee  = False
cfg_11_asks.verify_full_depth = False
lat_asks_11, rr_11 = benchmark_condition(
    model, tokenizer, _wc_probs_padded, cfg_11_asks, ARCH,
    disable_reuse=False,
    n_runs=N_RUNS, label="  §11 ASKS")

lat_vllm_11 = np.asarray(lat_vllm_11)
lat_asks_11 = np.asarray(lat_asks_11)
sp_11 = float(lat_vllm_11.mean() / lat_asks_11.mean())
_rng = np.random.default_rng(42)
_n11 = len(lat_vllm_11)
_boots_11 = [np.mean(lat_vllm_11[_rng.integers(0,_n11,_n11)])
             / np.mean(lat_asks_11[_rng.integers(0,_n11,_n11)])
             for _ in range(2000)]
ci11_lo, ci11_hi = float(np.percentile(_boots_11, 2.5)), float(np.percentile(_boots_11, 97.5))
_, p_11 = paired_ttest(lat_vllm_11, lat_asks_11) if _n11 > 1 else (0, float('nan'))

print(f"  No-KV / vLLM-equiv: {lat_vllm_11.mean():.1f} ± {lat_vllm_11.std():.1f} ms")
print(f"  ASKS (τ={tau_exact}):        {lat_asks_11.mean():.1f} ± {lat_asks_11.std():.1f} ms")
print(f"  ASKS vs No-KV speedup: {sp_11:.3f}× [95%CI {ci11_lo:.3f}–{ci11_hi:.3f}]  p={p_11:.3g}")
print(f"  ASKS reuse rate: {np.mean(rr_11)*100:.1f}%  (vs token-exact=0% on diverse wrappers)")
if ci11_lo > 1.0:
    print(f"  ✓ ASKS statistically faster than No-KV on padded diverse phrasings.")
elif ci11_hi < 1.0:
    print(f"  ✗ ASKS slower — prefix overhead exceeds saving at this n.")
else:
    print(f"  ⚠ Speedup within noise — see §9 n=512 result for cleaner evidence.")


In [ ]:
# ── Plot: exact-match vs ASKS reuse rates and similarity distribution ──────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(N_EXACT)
w = 0.35
ax.bar(x - w/2, df_cmp["exact_reuse_rate"], width=w, label="Token-exact",
       color="#aec6e8", edgecolor="white", alpha=0.85)
ax.bar(x + w/2, df_cmp["asks_reuse_rate"],  width=w, label=f"ASKS (τ={tau_exact})",
       color="#1f77b4", edgecolor="white", alpha=0.85)
ax.set_xlabel("Problem index")
ax.set_ylabel("KV reuse rate")
ax.set_title("Per-problem KV reuse: token-exact vs ASKS\n(diverse phrasing branches)")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
ax.axhline(1.0, color="#aaa", linestyle=":", linewidth=1)
# Aggregate annotation
_em_mean = df_cmp["exact_reuse_rate"].mean()
_as_mean = df_cmp["asks_reuse_rate"].mean()
ax.text(0.98, 0.97,
        f"Mean exact: {_em_mean:.0%}\nMean ASKS: {_as_mean:.0%}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.8))

ax = axes[1]
ax.scatter(range(len(df_cmp)), df_cmp["mean_asks_sim"], color="#1f77b4",
           s=60, zorder=5, label="Mean ASKS σ (branches 1-7)")
ax.axhline(tau_exact, color="red", linestyle="--", linewidth=1.5,
           label=f"τ = {tau_exact}")
ax.fill_between(range(len(df_cmp)),
                [tau_exact]*len(df_cmp), df_cmp["mean_asks_sim"],
                where=df_cmp["mean_asks_sim"] >= tau_exact,
                alpha=0.15, color="#1f77b4", label="Reuse zone")
ax.fill_between(range(len(df_cmp)),
                df_cmp["mean_asks_sim"], [tau_exact]*len(df_cmp),
                where=df_cmp["mean_asks_sim"] < tau_exact,
                alpha=0.15, color="#d62728", label="Recompute zone")
ax.set_xlabel("Problem index")
ax.set_ylabel("Mean ASKS cosine similarity σ")
ax.set_title("ASKS similarity vs threshold\n(diverse phrasing branches)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(max(0.75, df_cmp["mean_asks_sim"].min() - 0.02), 1.005)

plt.suptitle(
    f"Token-Exact vs ASKS Semantic KV Sharing — {MODEL_ID.split('/')[-1]} "
    f"(N={N_EXACT}, diverse phrasings)",
    fontsize=11)
plt.tight_layout()
plt.savefig("fig_exact_vs_asks.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_exact_vs_asks.pdf")


In [ ]:
# ── Consolidated experiment summary (§8–§12) ───────────────────────
print('=' * 75)
print('SUPPLEMENTARY EXPERIMENT SUMMARY')
print('=' * 75)

print('\n§8a  ASKS Semantic-Divergence Stress Test')
print(f'  N={N_SIM_PROBLEMS} problems × {len(_PHRASING_TEMPLATES)} phrasings')
for lvl in sorted(_DIVERGENCE_LEVELS):
    sub = df_sim[df_sim['divergence_level'] == lvl]
    print(f'  Level {lvl} ({_DIV_LABELS[lvl]:<22}): '
          f'σ = {sub["similarity"].mean():.4f}  '
          f'reuse = {sub["would_reuse_kv"].mean():.1%}')
print(f'  Empirical break-even reuse fraction: ρ* ≈ {_rho_star:.1%}')

print('\n§8b  τ Sweep (wall-clock; diverse short-prefix regime)')
for tau_b in TAUS_TO_BENCH:
    r  = bench_tau_results[tau_b]
    ci = r['ci']
    tag = '✓ net gain' if r['speedup'] >= 1.0 else '✗ net loss (ρ below break-even)'
    print(f'  τ={tau_b}: {r["speedup"]:.3f}×  '
          f'[95%CI {ci[0]:.3f}–{ci[1]:.3f}]  '
          f'reuse={r["reuse"]:.1%}  p={r["p"]:.2e}  [{tag}]')

print('\n§8c  Production (n≈512, identical prefix, ASKS vs No-KV)')
if 'results_8c' in dir():
    tag = '✓ NET GAIN' if results_8c['speedup'] > 1.0 else '✗ net loss'
    print(f'  Speedup: {results_8c["speedup"]:.3f}×  [{tag}]')
    print(f'  ASKS KV reuse: {results_8c["reuse"]:.1%}')
    print('  NOTE: equal to vLLM token-exact caching in this regime — see §11')

print('\n§9  Prefix-length sweep (end-to-end; ASKS)')
for n, r in sorted(sweep_results.items()):
    ci = r['ci_asks']
    print(f'  n={n:>4} tok: {r["speedup"]:.3f}×  '
          f'[95%CI {ci[0]:.3f}–{ci[1]:.3f}]  '
          f'reuse={r["reuse_rate"]:.1%}')

print('\n§10  RSBCM Validation (max_blocks=4 < B=8)')
expected = N_RSBCM_PROBS * (cfg_bm.branching_factor - 4)
print(f'  Expected evictions: {expected}   Observed: {total_ev_yes}')
print(f'  Overhead: {overhead:+.1f} ms/problem   '
      f'Agreement: {agreement_rate:.1%}   '
      f'{"✓ zero regressions" if agreement_rate >= 0.95 else "⚠ regressions"}')

print('\n§11  Token-Exact vs ASKS (diverse phrasings)')
print(f'  Token-exact reuse: {df_cmp["exact_reuse_rate"].mean():.1%}   '
      f'ASKS reuse: {df_cmp["asks_reuse_rate"].mean():.1%}   '
      f'ASKS marginal: '
      f'+{df_cmp["asks_reuse_rate"].mean() - df_cmp["exact_reuse_rate"].mean():.1%}')
if 'sp_11' in dir():
    print(f'  ASKS vs No-KV wall-clock: {sp_11:.3f}×  '
          f'[95%CI {ci11_lo:.3f}–{ci11_hi:.3f}]  p={p_11:.3g}')

print('\n§12  CGEE cross-dataset  (per-dataset θ calibration)')
if 'theta_sweep_results' in dir():
    for ds, r in theta_sweep_results.items():
        ok = r['n_gate_fired'] >= 2
        tag = '✓ gate active' if ok else '⚠ gate silent (θ mis-calibrated for dataset)'
        print(f'  {ds:<14}: θ={r["theta"]:.2f}  '
              f'fires={r["n_gate_fired"]}  '
              f'agree={r["agree"]:.1%}  [{tag}]')
    print('  Conclusion: θ is dataset-specific, not a universal constant — '
          'calibrate once per target workload.')
print('\n' + '=' * 75)


## §13  Camera-Ready Claims Checklist (v3 additions)

Every claim below has a corresponding empirical measurement with 95% CI.
Claims where the CI includes 1.0× are flagged as inconclusive.


In [ ]:
# ── Camera-ready claims checklist ──────────────────────────────
print('=' * 75)
print('CAMERA-READY CLAIMS CHECKLIST  (v5)')
print('=' * 75)


def _claim(label, sp, ci, p=None, note=''):
    lo, hi = ci
    if np.isnan(lo) or np.isnan(hi):
        status = '⚠ MISSING CI'
    elif lo > 1.0:
        status = '✓ SIGNIFICANT'
    elif hi < 1.0:
        status = '✗ SLOWER'
    else:
        status = '⚠ INCONCLUSIVE'
    p_str = f'  p={p:.3g}' if p is not None and np.isfinite(p) else ''
    print(f'  {label}')
    print(f'    {sp:.3f}× [{lo:.3f}–{hi:.3f}]{p_str}  → {status}')
    if note:
        print(f'    Note: {note}')


print('\n§6  Main benchmark (n=1024, identical prefix, B=8):')
if 'ci_rksc' in dir():
    sp_rksc_, _ = speedup(lat_rksc, lat_batch_nokv)
    _claim('RKSC (KV+CGEE) vs No-KV', sp_rksc_, ci_rksc, p_rksc)

print('\n§6b  CGEE verify-only isolation (decode-dominated regime):')
if 'ci_ver' in dir():
    _claim('CGEE verify-only', sp_ver, ci_ver, p_ver,
           note=f'mean exit layer {mean_exit_lyr:.1f}/{n_layers_arch}, '
                f'fire rate {fire_rate:.0%}')

print('\n§8b  τ sweep (wall-clock; diverse short-prefix regime):')
for tau_b in TAUS_TO_BENCH:
    r  = bench_tau_results[tau_b]
    ci = r.get('ci', (float('nan'), float('nan')))
    _claim(f'τ={tau_b} diverse', r['speedup'], ci, r.get('p'),
           note=f'ρ={r["reuse"]:.1%}, ρ*={_rho_star:.1%}')

print('\n§8c  Production (n≈512, identical prefix, vs vLLM):')
if 'results_8c' in dir():
    ci_8c = results_8c.get('speedup_ci', (float('nan'), float('nan')))
    _claim('ASKS vs No-KV', results_8c['speedup'], ci_8c,
           note='≡ vLLM token-exact in identical-prefix regime')

print('\n§9  Prefix-length sweep (identical prefix):')
if 'sweep_results' in dir():
    for n, r in sorted(sweep_results.items()):
        _claim(f'n={n}', r['speedup'],
                r.get('ci_asks', (float('nan'), float('nan'))))

print('\n§11  ASKS novelty over vLLM (diverse phrasings):')
print(f'  Reuse-rate gain: '
      f'+{df_cmp["asks_reuse_rate"].mean() - df_cmp["exact_reuse_rate"].mean():.1%} '
      f'pp over token-exact')
if 'sp_11' in dir():
    _claim('ASKS vs vLLM wall-clock', sp_11, (ci11_lo, ci11_hi), p_11)

print('\n§12  CGEE cross-dataset (per-dataset θ required):')
if 'theta_sweep_results' in dir():
    for ds, r in theta_sweep_results.items():
        if r['n_gate_fired'] >= 2:
            print(f'  {ds}: agree={r["agree"]:.1%} '
                  f'(gate fired {r["n_gate_fired"]}× at θ={r["theta"]:.2f})')
        else:
            print(f'  {ds}: INSUFFICIENT gate activity — needs per-dataset θ')

print('\n' + '=' * 75)
print('RKSC-core components validated:')
print('  ASKS   — §6, §8a, §8b, §8c, §9, §11')
print('  RSBCM  — §10')
print('  CGEE   — §6b (isolated), §12 (cross-dataset)')
print('LSKV removed from v5: identical to ASKS in identical-prefix regime.')
print('=' * 75)
